In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import json
from itertools import product
from pathlib import Path
from itertools import product
import hashlib

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')


In [ ]:
EXPERIMENT_NUMBER = 7

# -------------------------------------------------
# Region code mapping (single source of truth)
# -------------------------------------------------
REGION_CODE_MAP = {
    # "CH": "11_CH",
    "NOR": "08_NOR",
    "ISL": "06_ISL",
    "CEU": "11_CEU",
    "USCA": "01_CAW",  # Western Canada & US
}

REGION_GROUPS = {"CEU": ["FR", "CH", "IT_AT"], "USCA": ["ALA", "CAW"]}

SOURCE_CODES = list(REGION_CODE_MAP.keys())
TARGET_CODES = list(REGION_CODE_MAP.keys())

MODEL = "AdapterLSTM"

BASE_DIR = Path(cfg.dataPath) / path_cache / f"{MODEL}_{EXPERIMENT_NUMBER}"

MODELS_DIR = BASE_DIR / "models"
CACHE_DIR = BASE_DIR / "cache"
RESULTS_DIR = BASE_DIR / "results"

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# All directed source → target pairs (exclude same)
# -------------------------------------------------
PAIRS = [(src, tgt) for src, tgt in product(SOURCE_CODES, TARGET_CODES)
         if src != tgt]

# remove the pair (ISL, CEU) because it is too hard to learn and causes instability in training
# PAIRS.remove(("CH", "CEU"))
# PAIRS.remove(("CEU", "CH"))

PAIR_KEYS = [f"{src}_to_{tgt}" for src, tgt in PAIRS]
# PAIRS.remove(("CH", "CEU"))
# PAIRS.remove(("CEU", "CH"))

print(f"Pairs: (N = {len(PAIRS)})", PAIRS)
print("Pair keys:", PAIR_KEYS)

In [ ]:
def resolve_region_codes(region, region_groups=None):
    if region_groups is None:
        region_groups = {}
    return region_groups.get(region, [region])


def filter_by_region(df, region, region_groups=None, source_col="SOURCE_CODE"):
    codes = resolve_region_codes(region, region_groups)
    return df[df[source_col].isin(codes)].copy()


def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode("utf-8")).hexdigest()[:10]

## Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
for rid, df in dfs.items():
    n_rows = len(df)
    n_nan = df.isna().sum().sum()
    print(f"\nRegion {rid}")
    print("rows:", n_rows)
    print("total NaNs:", n_nan)

    if n_nan > 0:
        print("columns with NaNs:")
        print(df.isna().sum()[df.isna().sum() > 0].sort_values(
            ascending=False).head(20))

## Monthly datasets:

In [ ]:
# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

MONTHLY_COLS = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
    "ELEVATION_DIFFERENCE",
]
STATIC_COLS = ["aspect", "slope", "svf"]
FEATURE_COLS = MONTHLY_COLS + STATIC_COLS

In [ ]:
# load all stake dfs once
dfs = {
    rid: load_stakes_for_rgi_region(cfg, rid)
    for rid in tqdm(RGI_REGIONS.keys(), desc="Loading stake regions")
}

# recompute only pairs involving these symbolic/original region labels
# examples:
#   None          -> load all
#   {"CEU"}       -> recompute all pairs where src=="CEU" or tgt=="CEU"
#   {"ISL"}       -> recompute all pairs involving ISL
#   {"CEU","ISL"} -> recompute all pairs involving either CEU or ISL
#   "ALL"         -> recompute all
RECOMPUTE_REGIONS = None

all_pair_res = {}
all_pair_split_info = {}


def resolve_region(code, region_groups):
    """Return either the raw code or the grouped list of codes."""
    return region_groups.get(code, code)


def should_recompute_pair(src, tgt, recompute_regions=None):
    """
    Decide whether a pair should be recomputed.

    Parameters
    ----------
    src, tgt : str
        Pair labels as they appear in PAIRS, e.g. "CH", "ISL", "CEU".
    recompute_regions : set[str] | None | str
        - None  -> load all pairs
        - "ALL" -> recompute all pairs
        - set   -> recompute only pairs where src or tgt is in this set
    """
    if recompute_regions is None:
        return False
    if recompute_regions == "ALL":
        return True
    return (src in recompute_regions) or (tgt in recompute_regions)


for src, tgt in tqdm(PAIRS, desc="Preparing pairwise monthlies"):
    key = f"{src}_to_{tgt}"

    src_codes = resolve_region(src, REGION_GROUPS)
    tgt_codes = resolve_region(tgt, REGION_GROUPS)

    recompute_this = should_recompute_pair(
        src=src,
        tgt=tgt,
        recompute_regions=RECOMPUTE_REGIONS,
    )

    res_xreg, split_info = prepare_monthly_df_xreg_pairwise(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        source_code=src_codes,
        target_code=tgt_codes,
        run_flag=recompute_this,
        region_name=f"XREG_{src}_TO_{tgt}",
        csv_subfolder=f"CrossRegional/{src}_to_{tgt}/csv",
    )

    all_pair_res[key] = res_xreg
    all_pair_split_info[key] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    status = "recomputed" if recompute_this else "loaded"

    print("\n" + "=" * 80)
    print(f"Pair: {key} [{status}]")
    print(f"Train glaciers ({src}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers ({tgt}):", len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "| Test rows:", len(df_test))

In [ ]:
df_sc_check = summarize_source_code_issues_in_all_pair_res(all_pair_res)
df_sc_check

In [ ]:
def summarize_nans(df, name, top_n=20):
    total_nans = int(df.isna().sum().sum())
    print(f"\n--- {name} ---")
    print(f"rows={len(df)}, cols={df.shape[1]}, total_nans={total_nans}")

    if total_nans == 0:
        print("No NaNs")
        return

    nan_by_col = df.isna().sum()
    nan_by_col = nan_by_col[nan_by_col > 0].sort_values(ascending=False)

    print("Top columns with NaNs:")
    print(nan_by_col.head(top_n))

    if "SOURCE_CODE" in df.columns:
        bad_rows = df[df[nan_by_col.index].isna().any(axis=1)]
        print("\nBad rows by SOURCE_CODE:")
        print(bad_rows["SOURCE_CODE"].value_counts(dropna=False))

    if "GLACIER" in df.columns:
        bad_rows = df[df[nan_by_col.index].isna().any(axis=1)]
        print("\nBad rows by GLACIER:")
        print(bad_rows["GLACIER"].value_counts(dropna=False).head(15))


for key, res_xreg in all_pair_res.items():
    print("\n" + "=" * 100)
    print(f"PAIR {key}")

    for subkey in ["df_train", "df_train_aug", "df_test", "df_test_aug"]:
        if subkey in res_xreg:
            summarize_nans(res_xreg[subkey], f"{key} | {subkey}")

## Experiment design:

In [ ]:
for key, res in all_pair_res.items():
    src_label, tgt_label = key.split("_to_")

    src_codes = resolve_region_codes(src_label, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt_label, REGION_GROUPS)

    df_test = res["df_test"]
    df_train = res["df_train"]

    # In pairwise setup, df_test should already be target-only and df_train source-only,
    # but we still filter explicitly for robustness.
    df_tgt = df_test[df_test["SOURCE_CODE"].isin(tgt_codes)].copy()
    df_src = df_train[df_train["SOURCE_CODE"].isin(src_codes)].copy()

    print("\n" + "=" * 80)
    print(f"Pair: {src_label} → {tgt_label}")

    print(f"{src_label} monthly rows:", len(df_src))
    print(f"{src_label} glaciers:", df_src["GLACIER"].nunique())
    if len(df_src) > 0:
        print(
            f"{src_label} year range:",
            df_src["YEAR"].min(),
            "-",
            df_src["YEAR"].max(),
        )
    else:
        print(f"No source rows found for {src_label}.")

    print(f"{tgt_label} monthly rows:", len(df_tgt))
    print(f"{tgt_label} glaciers:", df_tgt["GLACIER"].nunique())
    if len(df_tgt) > 0:
        print(
            f"{tgt_label} year range:",
            df_tgt["YEAR"].min(),
            "-",
            df_tgt["YEAR"].max(),
        )
    else:
        print(f"No target rows found for {tgt_label}.")

### Fixed glacier hold-out split (spatial generalization):
#### Calculate directional “shift as seen from the source model

In [ ]:
EXCLUDE_COLS = ['ELEVATION_DIFFERENCE']  # or [] to disable
effective_feature_cols = [f for f in FEATURE_COLS if f not in EXCLUDE_COLS]

# Fit one global scaler across CH, NOR, ISL
X_global_list = []
for key, res in all_pair_res.items():
    X_train = to_feature_matrix(res["df_train"],
                                effective_feature_cols,
                                dropna=True)
    if X_train.size > 0:
        X_global_list.append(X_train)
    X_test = to_feature_matrix(res["df_test"],
                               effective_feature_cols,
                               dropna=True)
    if X_test.size > 0:
        X_global_list.append(X_test)

X_global = np.vstack(X_global_list)
global_scaler = StandardScaler()
global_scaler.fit(X_global)
print("Global scaler fitted on:", X_global.shape)
print("Features used:", effective_feature_cols)
print(global_scaler.mean_)
print(global_scaler.scale_)

In [ ]:
pair_splits = {}
pbar = tqdm(list(all_pair_res.items()), desc="Pool/holdout + distances")
for key, res in pbar:
    src, tgt = key.split("_to_")
    pbar.set_postfix_str(key)

    src_codes = resolve_region_codes(src, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt, REGION_GROUPS)

    df_src = res["df_train"].copy()
    df_src = df_src[df_src["SOURCE_CODE"].isin(src_codes)].copy()
    df_tgt = res["df_test"].copy()
    df_tgt = df_tgt[df_tgt["SOURCE_CODE"].isin(tgt_codes)].copy()

    if len(df_src) == 0 or len(df_tgt) == 0:
        print(f"Skipping pair {key} due to empty source or target. "
              f"src_codes={src_codes}, tgt_codes={tgt_codes}")
        pair_splits[key] = {
            "source": src,
            "target": tgt,
            "error": "empty src or tgt"
        }
        continue

    n_clusters = min(6, df_tgt.GLACIER.nunique())
    df_pool, df_holdout, holdout_glaciers, pool_glaciers, gfeat, split_summary = (
        holdout_split_cluster_stratified(
            df_tgt,
            holdout_frac=0.1,
            seed=cfg.seed,
            n_clusters=n_clusters,
        ))
    print(key, df_holdout.GLACIER.unique())
    dist = compute_topoclimatic_distances(
        df_src=df_src,
        df_pool=df_pool,
        df_holdout=df_holdout,
        feature_cols=effective_feature_cols,
        scaler=global_scaler,
        seed=cfg.seed,
        energy_max_samples=4000,
        topo_cols=STATIC_COLS,
        climate_cols=MONTHLY_COLS,
    )

    pair_splits[key] = {
        "source": src,
        "target": tgt,
        "df_src": df_src,
        "df_target": df_tgt,
        "df_pool": df_pool,
        "df_holdout": df_holdout,
        "pool_glaciers": pool_glaciers,
        "holdout_glaciers": holdout_glaciers,
        "gfeat": gfeat,
        "split_summary": split_summary,
        "distances": dist,
    }

    print(
        f"{key}: "
        f"D_energy(src, tgt_all)={dist['D_energy_src_tgt_all']:.3f} | "
        f"D_wass(src, tgt_all)={dist['D_wass_src_tgt_all']:.3f} | "
        f"D_energy(src, hold)={dist['D_energy_src_holdout']:.3f} | "
        f"D_energy(pool, hold)={dist['D_energy_pool_holdout']:.3f} | "
        f"D_energy(src, tgt_all)[topo]={dist.get('D_energy_src_tgt_all_topo', float('nan')):.3f} | "
        f"D_energy(src, tgt_all)[clim]={dist.get('D_energy_src_tgt_all_clim', float('nan')):.3f} | "
    )

#### Plot location:

In [ ]:
# --- 1) One place to define the per-target region metadata you need for mapping ---
TARGET_REGION_META = {
    "ISL": {
        "rgi_region_id": "06",
        "outline_shp_rel": "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
        "title": "Glacier PMB locations Iceland",
        "extent": (-25, -11, 62, 68),
    },
    "NOR": {
        "rgi_region_id": "08",
        "outline_shp_rel":
        "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
        # If you actually use the Norway-only file in your repo, swap path accordingly.
        "title": "Glacier PMB locations Norway",
        # rough bbox for mainland Norway + Svalbard excluded; adjust if needed
        "extent": (4, 32, 57, 72),
    },
    "CH": {
        "rgi_region_id": "11",
        "outline_shp_rel":
        "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
        # If you have a Switzerland-only outline, swap path accordingly.
        "title": "Glacier PMB locations Switzerland",
        # rough bbox for Switzerland
        "extent": (5.8, 13.7, 44.5, 47.9),
    },
    "CEU": {
        "rgi_region_id": "11",
        "outline_shp_rel":
        "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
        # If you have a Switzerland-only outline, swap path accordingly.
        "title": "Glacier PMB locations Central Europe (FR+CH+IT+AT)",
        # rough bbox for CEU
        "extent": (5.8, 13.7, 44.5, 47.9),
    },
    "USCA": {
        "rgi_region_id": ["01", "02"],
        "outline_shp_rel": "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
        # If you have a Switzerland-only outline, swap path accordingly.
        "title": "Glacier PMB locations US CA (ALA+CAW)",
        # rough bbox for CEU
        "extent": (5.8, 13.7, 44.5, 47.9),
    },
}


# --- 2) A small helper to run the exact same plotting pipeline for any target ---
def make_pool_holdout_map_for_target(
        cfg,
        target_code: str,
        holdout_glaciers,
        pool_glaciers=None,  # optional, not strictly needed if labels are inferred elsewhere
        split_name: str = "spatial",
        sizes=(100, 1500),
        size_legend_values=(30, 100, 1000),
        cmap_for_train=cm.batlow,
):
    meta = TARGET_REGION_META[target_code]

    ft_glaciers_by_split = {split_name: holdout_glaciers}

    data_tgt, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
        cfg,
        rgi_region_id=meta["rgi_region_id"],
        outline_shp_path=str(Path(cfg.dataPath) / meta["outline_shp_rel"]),
        ft_glaciers_by_split=ft_glaciers_by_split,
        split_names=[split_name],
        ft_label_col="Pool/Hold-out glacier",
        ft_label_ft="Pool",
        ft_label_holdout="Hold-out",
    )

    glacier_df = glacier_info_by_split[split_name]

    # colors (keep your existing style)
    try:
        colors = get_cmap_hex(cmap_for_train, 10)  # noqa: F821
        train_color = colors[0]
    except Exception:
        train_color = "#1f4e79"  # fallback

    palette = {"Hold-out": train_color, "Pool": "#b2182b"}

    fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
        glacier_info=glacier_df,
        glacier_outline_rgi=glacier_outline_rgi,
        title=meta["title"],
        extent=meta["extent"],
        sizes=sizes,
        size_legend_values=size_legend_values,
        palette=palette,
        cmap_for_train=cmap_for_train,
        split_col="Pool/Hold-out glacier",
    )

    return {
        "data": data_tgt,
        "glacier_outline_rgi": glacier_outline_rgi,
        "glacier_df": glacier_df,
        "fig": fig,
        "ax": ax,
        "glacier_info_plot": glacier_info_plot,
        "scaled_size_fn": scaled_size_fn,
        "palette": palette,
    }

In [ ]:
# --- 3) Example usage for any pair you already split ---
# pick a pair key, e.g. "CH_to_ISL"
key = "CEU_to_ISL"
tgt = key.split("_to_")[1]

holdout_glaciers = pair_splits[key]["holdout_glaciers"]
pool_glaciers = pair_splits[key]["pool_glaciers"]

out = make_pool_holdout_map_for_target(
    cfg=cfg,
    target_code=tgt,
    holdout_glaciers=holdout_glaciers,
    pool_glaciers=pool_glaciers,
)

In [ ]:
# --- 3) Example usage for any pair you already split ---
# pick a pair key, e.g. "CH_to_ISL"
key = "CEU_to_NOR"
tgt = key.split("_to_")[1]

holdout_glaciers = pair_splits[key]["holdout_glaciers"]
pool_glaciers = pair_splits[key]["pool_glaciers"]

out = make_pool_holdout_map_for_target(
    cfg=cfg,
    target_code=tgt,
    holdout_glaciers=holdout_glaciers,
    pool_glaciers=pool_glaciers,
)

In [ ]:
# --- 3) Example usage for any pair you already split ---
# pick a pair key, e.g. "CH_to_ISL"
key = "ISL_to_CEU"
tgt = key.split("_to_")[1]

holdout_glaciers = pair_splits[key]["holdout_glaciers"]
pool_glaciers = pair_splits[key]["pool_glaciers"]

out = make_pool_holdout_map_for_target(
    cfg=cfg,
    target_code=tgt,
    holdout_glaciers=holdout_glaciers,
    pool_glaciers=pool_glaciers,
)

### Build static assets once (source + target holdout)

In [ ]:
static_assets_by_pair = {}
errors_static_assets = {}

pbar = tqdm(list(pair_splits.items()), desc="Building static TL assets")
for key, d in pbar:
    if "error" in d:
        errors_static_assets[key] = d["error"]
        continue

    src_label = d["source"]
    tgt_label = d["target"]

    src_codes = resolve_region_codes(src_label, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt_label, REGION_GROUPS)

    print(src_codes, tgt_codes)

    pbar.set_postfix_str(f"{src_label}->{tgt_label}")

    res_xreg = all_pair_res[key]
    holdout_glaciers = d["holdout_glaciers"]

    sig = split_signature(holdout_glaciers)
    cache_dir_pair = CACHE_DIR / key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    def _build(force_recompute: bool):
        return build_static_tl_assets_source_and_holdout(
            cfg=cfg,
            res_xreg=res_xreg,
            target_codes=tgt_codes,
            source_codes=src_codes,
            target_label=tgt_label,
            source_label=src_label,
            holdout_glaciers=holdout_glaciers,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=cache_dir_pair,
            force_recompute=force_recompute,
            show_progress=False,
        )

    try:
        force_recompute = False
        static_assets = _build(force_recompute=force_recompute)
        static_assets_by_pair[key] = static_assets

    except Exception as e:
        msg = repr(e)
        should_retry = ("Missing SOURCE_CODE for ds key" in msg) or isinstance(
            e, KeyError)

        if should_retry:
            try:
                static_assets = _build(force_recompute=True)
                static_assets_by_pair[key] = static_assets
            except Exception as e2:
                errors_static_assets[
                    key] = f"initial={msg} | recompute_failed={repr(e2)}"
        else:
            errors_static_assets[key] = msg

print(errors_static_assets)

### Define the pools for experiments:

In [ ]:
pool_capacity_by_pair = {}
errors_pool_capacity = {}

pbar = tqdm(list(pair_splits.items()), desc="Computing pool capacity (G/Y/M)")
for key, d in pbar:
    if "error" in d:
        errors_pool_capacity[key] = d["error"]
        continue

    src = d["source"]
    tgt = d["target"]
    pbar.set_postfix_str(f"{src}->{tgt}")

    tgt_codes = resolve_region_codes(tgt, REGION_GROUPS)
    pool_glaciers = d["pool_glaciers"]

    # monthly target df (already filtered to target in earlier steps)
    df_tgt_monthly = d["df_target"]
    df_pool_monthly = df_tgt_monthly[df_tgt_monthly["GLACIER"].isin(
        pool_glaciers)].copy()

    if len(df_pool_monthly) == 0:
        errors_pool_capacity[key] = "empty df_pool_monthly"
        continue

    G_max = int(df_pool_monthly["GLACIER"].nunique())
    Y_max = int(df_pool_monthly.groupby("GLACIER")["YEAR"].nunique().max())

    # raw stake df for the target RGI region
    rid = TARGET_REGION_META[tgt]["rgi_region_id"]

    # ensure rid is always iterable
    if not isinstance(rid, (list, tuple)):
        rid = [rid]

    df_tgt_raw = pd.concat([dfs[r].copy() for r in rid], ignore_index=True)
    df_tgt_raw = df_tgt_raw[df_tgt_raw["SOURCE_CODE"].isin(tgt_codes)].copy()
    df_tgt_raw = df_tgt_raw[df_tgt_raw["GLACIER"].isin(pool_glaciers)].copy()

    if len(df_tgt_raw) == 0:
        errors_pool_capacity[
            key] = f"empty raw df for rid={rid}, tgt_codes={tgt_codes} after filtering pool_glaciers"
        continue

    rows_per_gy = df_tgt_raw.groupby(["GLACIER", "YEAR"]).size()
    M_p25 = int(rows_per_gy.quantile(0.25))
    M_p50 = int(rows_per_gy.median())
    M_p90 = int(rows_per_gy.quantile(0.90))
    M_max = int(rows_per_gy.max())
    M_min = int(rows_per_gy.min())

    pool_capacity_by_pair[key] = {
        "source": src,
        "target": tgt,
        "rid": rid,
        "target_codes": tgt_codes,
        "G_max": G_max,
        "Y_max": Y_max,
        "M_min": M_min,
        "M_p25": M_p25,
        "M_p50": M_p50,
        "M_p90": M_p90,
        "M_max": M_max,
    }

    print("\n" + "=" * 80)
    print(f"POOL CAPACITY | {key}")
    print("target_codes:", tgt_codes)
    print("G_max:", G_max)
    print("Y_max (max years on a glacier):", Y_max)
    print("Rows per glacier-year (raw format): min", M_min, "| p25", M_p25,
          "| median", M_p50, "| p90", M_p90, "| max", M_max)

# optional: tabular summary
df_pool_capacity = pd.DataFrame.from_dict(
    pool_capacity_by_pair,
    orient="index").reset_index().rename(columns={"index": "pair"})
display(df_pool_capacity)

if errors_pool_capacity:
    print("\nErrors:")
    for k, v in errors_pool_capacity.items():
        print(" ", k, ":", v)

In [ ]:
sanity_samples_by_pair = {}
errors_sanity_samples = {}

# choose a single (G, Y) for the sanity check
G_sanity = 5
Y_sanity = 5

pbar = tqdm(list(pair_splits.items()),
            desc="Sanity-check sampling (one budget)")
for key, d in pbar:
    if "error" in d:
        errors_sanity_samples[key] = d["error"]
        continue

    src = d["source"]
    tgt = d["target"]
    pbar.set_postfix_str(f"{src}->{tgt}")

    df_tgt_monthly = d["df_target"]
    pool_glaciers = d["pool_glaciers"]

    # pool rows (monthly) for this target
    df_pool_rows = df_tgt_monthly[df_tgt_monthly["GLACIER"].isin(
        pool_glaciers)].copy()
    if len(df_pool_rows) == 0:
        errors_sanity_samples[key] = "empty df_pool_rows"
        continue

    # cap G/Y so the sanity check doesn't crash on small targets
    G_eff = min(G_sanity, int(df_pool_rows["GLACIER"].nunique()))
    Y_eff = min(Y_sanity,
                int(df_pool_rows.groupby("GLACIER")["YEAR"].nunique().max()))

    try:
        df_ft, chosen, picked_years = sample_monitoring_subset_from_pool(
            df_pool=df_pool_rows,
            G=G_eff,
            Y=Y_eff,
            seed=(abs(hash(key)) + int(cfg.seed)) %
            (2**32 - 1),  # pair-specific reproducible
            glacier_pick_method="random",
            year_pick_method="earliest_block",
            enforce_full_Y_if_possible=True,
        )

        sanity_samples_by_pair[key] = {
            "df_ft": df_ft,
            "chosen_glaciers": chosen,
            "picked_years": picked_years,
            "G_used": G_eff,
            "Y_used": Y_eff,
        }

        print("\n" + "=" * 90)
        print(f"Sanity check | {key} | G={G_eff}, Y={Y_eff}")
        print("Chosen glaciers:", sorted(chosen))
        print("Years per glacier:")
        for gid in sorted(chosen):
            print(" ", gid, picked_years.get(gid))

    except Exception as e:
        errors_sanity_samples[key] = repr(e)

if errors_sanity_samples:
    print("\nErrors:")
    for k, v in errors_sanity_samples.items():
        print(" ", k, ":", v)

### LSTM source baseline:

In [ ]:
log_path_gs_results = {
    "ISL":
    BASE_DIR / "GS_results/lstm_param_search_progress_OOS_ISL_2026-02-11.csv",
    "NOR":
    BASE_DIR / "GS_results/lstm_param_search_progress_OOS_NOR_2026-02-09.csv",
    "CH": BASE_DIR / "GS_results/lstm_param_search_progress_CH_2026-02-18.csv",
}

for country, path in log_path_gs_results.items():
    if not path.exists():
        print(f"WARNING: {country} log file not found: {path}")


In [ ]:
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)

params_by_key['11_CEU'] = params_by_key['11_CH']
params_by_key['01_USCA'] = default_params
params_by_key['01_CAW'] = default_params

baseline_models = {}
baseline_infos = {}

for src in tqdm(SOURCE_CODES, desc="Training/loading source baselines"):
    # pick any pair_key that uses this src, to get compatible tl_assets
    pair_key = next(k for k in static_assets_by_pair.keys()
                    if k.startswith(f"{src}_to_"))

    static_assets = static_assets_by_pair[pair_key]
    tl_assets_static = {"STATIC": static_assets}

    default_params = params_by_key[REGION_CODE_MAP[src]]
    print(default_params)

    model_src, model_path, info = train_or_load_source_baseline(
        cfg=cfg,
        tl_assets=tl_assets_static,
        default_params=default_params,
        device=device,
        source_code=src,
        models_dir=MODELS_DIR,
        prefix=f"lstm_{src}",  # e.g. lstm_CH
        key="BASELINE",
        train_flag=False,  # set False if you only want loading
        force_retrain=False,
        epochs=150,
        batch_size_train=64,
        batch_size_val=128,
        verbose=True,
        date="2026-03-18",  # or None
    )

    baseline_models[src] = model_src
    baseline_infos[src] = {
        "path": model_path,
        "info": info,
        "pair_key_used": pair_key
    }

    print(f"{src}: baseline -> {model_path} (assets from {pair_key})")

### Sample FT subsets & compute distances:

In [ ]:
# -----------------------
# Budgets & seeds (global)
# -----------------------
G_set = [5, 8, 10, 15, 20, 30]
Y_set = [5, 10, 15, 20, 30]
BUDGETS = [{"G": int(g), "Y": int(y)} for g in G_set for y in Y_set]

N_SEEDS = 20
SEEDS = list(range(1, N_SEEDS + 1))

print("Total budget points:", len(BUDGETS))
print(pd.DataFrame(BUDGETS).sort_values(["G", "Y"]).head(12))
print("Seeds:", SEEDS)

In [ ]:
def load_pair_distance_registry(cache_dir_pair):
    dist_registry_path = Path(cache_dir_pair) / "distances_registry.csv"
    if dist_registry_path.exists():
        df_dist_pair = pd.read_csv(dist_registry_path)
    else:
        df_dist_pair = pd.DataFrame()
    return df_dist_pair, dist_registry_path

def compute_one_distance_row(
    *,
    exp_key,
    pair_key,
    src,
    tgt,
    G,
    Y,
    seed,
    df_src,
    df_ft,
    df_holdout,
    FEATURE_COLS,
    global_scaler,
    STATIC_COLS,
    MONTHLY_COLS,
):
    dist = compute_topoclimatic_distances_sets(
        df_src=df_src,
        df_ft=df_ft,
        df_holdout=df_holdout,
        feature_cols=FEATURE_COLS,
        scaler=global_scaler,
        seed=seed,
        energy_max_samples=4000,
        exclude_cols=["ELEVATION_DIFFERENCE"],
        topo_cols=STATIC_COLS,
        climate_cols=MONTHLY_COLS,
    )

    return {
        "pair": pair_key,
        "source": src,
        "target": tgt,
        "exp_key": exp_key,
        "G": G,
        "Y": Y,
        "seed": seed,
        **dist,
    }

def load_or_compute_distances_from_pair_registry(
    *,
    df_dist_pair,
    exp_key,
    pair_key,
    src,
    tgt,
    G,
    Y,
    seed,
    df_src,
    df_ft,
    df_holdout,
    FEATURE_COLS,
    global_scaler,
    STATIC_COLS,
    MONTHLY_COLS,
    recompute_distances=False,
):
    if (not recompute_distances) and ("exp_key" in df_dist_pair.columns):
        hit = df_dist_pair[df_dist_pair["exp_key"] == exp_key]
        if len(hit) > 0:
            return hit.iloc[0].to_dict(), df_dist_pair

    dist_row = compute_one_distance_row(
        exp_key=exp_key,
        pair_key=pair_key,
        src=src,
        tgt=tgt,
        G=G,
        Y=Y,
        seed=seed,
        df_src=df_src,
        df_ft=df_ft,
        df_holdout=df_holdout,
        FEATURE_COLS=FEATURE_COLS,
        global_scaler=global_scaler,
        STATIC_COLS=STATIC_COLS,
        MONTHLY_COLS=MONTHLY_COLS,
    )

    df_dist_pair = pd.concat(
        [df_dist_pair, pd.DataFrame([dist_row])],
        axis=0,
        ignore_index=True,
    )

    df_dist_pair = df_dist_pair.drop_duplicates(subset=["exp_key"], keep="last")

    return dist_row, df_dist_pair

In [ ]:
def build_one_budget_experiment_light(
    *,
    cfg,
    pair_key,
    src,
    tgt,
    df_src,
    df_holdout,
    df_pool_rows,
    static_assets,
    res_xreg,
    cache_dir_pair,
    holdout_sig,
    G,
    Y,
    seed,
    MONTHLY_COLS,
    STATIC_COLS,
    FEATURE_COLS,
    global_scaler,
    df_dist_pair,
    recompute_distances=False,
):
    exp_key = f"TL_{src}_to_{tgt}_G{G}_Y{Y}_seed{seed}"

    # 1) deterministic sampling
    df_ft, chosen, _ = sample_monitoring_subset_from_pool(
        df_pool=df_pool_rows,
        G=G,
        Y=Y,
        seed=seed,
        glacier_pick_method="random",
        year_pick_method="earliest_block",
        enforce_full_Y_if_possible=True,
    )

    # 2) build/load FT dataset cache
    _ = build_budget_assets_finetune_only(
        cfg=cfg,
        res_xreg=res_xreg,
        static_assets=static_assets,
        df_ft=df_ft,
        exp_key=exp_key,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=cache_dir_pair,
        force_recompute=False,
        val_ratio=0.2,
        show_progress=False,
        seed=seed,
    )

    # 3) load or compute distances from pair registry
    dist_row, df_dist_pair = load_or_compute_distances_from_pair_registry(
        df_dist_pair=df_dist_pair,
        exp_key=exp_key,
        pair_key=pair_key,
        src=src,
        tgt=tgt,
        G=G,
        Y=Y,
        seed=seed,
        df_src=df_src,
        df_ft=df_ft,
        df_holdout=df_holdout,
        FEATURE_COLS=FEATURE_COLS,
        global_scaler=global_scaler,
        STATIC_COLS=STATIC_COLS,
        MONTHLY_COLS=MONTHLY_COLS,
        recompute_distances=recompute_distances,
    )

    row = {
        "pair": pair_key,
        "source": src,
        "target": tgt,
        "exp_key": exp_key,
        "G": G,
        "Y": Y,
        "seed": seed,
        "cache_dir": str(cache_dir_pair),
        "holdout_sig": holdout_sig,
        "n_rows_ft": int(len(df_ft)),
        "n_glaciers_ft": int(df_ft["GLACIER"].nunique()),
        "n_years_ft": int(df_ft["YEAR"].nunique()),
    }

    # attach only the distance columns, not duplicate metadata
    dist_only = {
        k: v for k, v in dist_row.items()
        if k not in {"pair", "source", "target", "exp_key", "G", "Y", "seed"}
    }
    row.update(dist_only)

    return row, df_dist_pair

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pandas as pd


def compute_one_distance_task(args):
    """
    Worker: compute distances for one experiment.
    Must stay top-level for multiprocessing to work cleanly in notebooks.
    """
    (
        pair_key,
        src,
        tgt,
        G,
        Y,
        seed,
        df_src,
        df_holdout,
        df_ft,
        FEATURE_COLS,
        global_scaler,
        STATIC_COLS,
        MONTHLY_COLS,
    ) = args

    exp_key = f"TL_{src}_to_{tgt}_G{G}_Y{Y}_seed{seed}"

    dist = compute_topoclimatic_distances_sets(
        df_src=df_src,
        df_ft=df_ft,
        df_holdout=df_holdout,
        feature_cols=FEATURE_COLS,
        scaler=global_scaler,
        seed=seed,
        energy_max_samples=4000,
        exclude_cols=["ELEVATION_DIFFERENCE"],
        topo_cols=STATIC_COLS,
        climate_cols=MONTHLY_COLS,
    )

    row = {
        "pair": pair_key,
        "source": src,
        "target": tgt,
        "exp_key": exp_key,
        "G": G,
        "Y": Y,
        "seed": seed,
        **dist,
    }
    return row


def get_cached_distance_row(df_dist_pair, exp_key):
    if len(df_dist_pair) == 0 or "exp_key" not in df_dist_pair.columns:
        return None
    hit = df_dist_pair[df_dist_pair["exp_key"] == exp_key]
    if len(hit) == 0:
        return None
    return hit.iloc[0].to_dict()

In [ ]:
registry_path = RESULTS_DIR / "experiment_registry_all_pairs.csv"
REBUILD_REGISTRY = False
RECOMPUTE_DISTANCES = False
N_JOBS = min(4, os.cpu_count() or 1)  # start modestly

if registry_path.exists() and not REBUILD_REGISTRY:
    df_existing = pd.read_csv(registry_path)
    experiment_rows = df_existing.to_dict(orient="records")
    done_exp_keys = set(df_existing["exp_key"].astype(str))
    print(f"Loaded existing registry with {len(done_exp_keys)} experiments")
else:
    experiment_rows = []
    done_exp_keys = set()

errors_assets_by_pair = {}
skipped_tasks_by_pair = {}

pairs_iter = tqdm(list(pair_splits.items()), desc="Pairs (build cache + registry)")

for pair_key, d in pairs_iter:
    if "error" in d:
        continue

    src = d["source"]
    tgt = d["target"]
    pairs_iter.set_postfix_str(f"{src}->{tgt}")

    if pair_key not in static_assets_by_pair:
        errors_assets_by_pair.setdefault(pair_key, []).append("missing static_assets_by_pair")
        continue

    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    sig = split_signature(d["holdout_glaciers"])
    cache_dir_pair = CACHE_DIR / pair_key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    # pair-level distance cache
    df_dist_pair, dist_registry_path = load_pair_distance_registry(cache_dir_pair)
    done_dist_exp_keys = set(df_dist_pair["exp_key"].astype(str)) if len(df_dist_pair) else set()

    df_src = d["df_src"]
    df_holdout = d["df_holdout"]

    pool_glaciers = d["pool_glaciers"]
    df_pool_rows = d["df_target"][d["df_target"]["GLACIER"].isin(pool_glaciers)].copy()

    if len(df_pool_rows) == 0:
        errors_assets_by_pair.setdefault(pair_key, []).append("empty df_pool_rows")
        continue

    G_max = int(df_pool_rows["GLACIER"].nunique())
    Y_max = int(df_pool_rows.groupby("GLACIER")["YEAR"].nunique().max())

    tasks = [(b, seed) for b in BUDGETS for seed in SEEDS]
    skipped = []
    errs = []

    # collected in main process
    pair_rows = []
    distance_jobs = []

    task_iter = tqdm(tasks, desc=pair_key, leave=False)

    for b, seed in task_iter:
        G = int(b["G"])
        Y = int(b["Y"])
        exp_key = f"TL_{src}_to_{tgt}_G{G}_Y{Y}_seed{seed}"

        if (G > G_max) or (Y > Y_max):
            skipped.append({
                "G": G,
                "Y": Y,
                "seed": seed,
                "reason": f"exceeds pool capacity (G_max={G_max}, Y_max={Y_max})",
            })
            continue

        try:
            # deterministic sample in main process
            df_ft, chosen, _ = sample_monitoring_subset_from_pool(
                df_pool=df_pool_rows,
                G=G,
                Y=Y,
                seed=seed,
                glacier_pick_method="random",
                year_pick_method="earliest_block",
                enforce_full_Y_if_possible=True,
            )

            # build/load dataset cache in main process only
            _ = build_budget_assets_finetune_only(
                cfg=cfg,
                res_xreg=res_xreg,
                static_assets=static_assets,
                df_ft=df_ft,
                exp_key=exp_key,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=cache_dir_pair,
                force_recompute=False,
                val_ratio=0.2,
                show_progress=False,
                seed=seed,
            )

            # lightweight metadata always prepared in main process
            meta_row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key,
                "G": G,
                "Y": Y,
                "seed": seed,
                "cache_dir": str(cache_dir_pair),
                "holdout_sig": sig,
                "n_rows_ft": int(len(df_ft)),
                "n_glaciers_ft": int(df_ft["GLACIER"].nunique()),
                "n_years_ft": int(df_ft["YEAR"].nunique()),
            }

            # fully cached case: reuse existing global + pair-distance registry
            if (
                exp_key in done_exp_keys
                and exp_key in done_dist_exp_keys
                and not RECOMPUTE_DISTANCES
            ):
                cached_dist_row = get_cached_distance_row(df_dist_pair, exp_key)
                if cached_dist_row is not None:
                    dist_only = {
                        k: v for k, v in cached_dist_row.items()
                        if k not in {"pair", "source", "target", "exp_key", "G", "Y", "seed"}
                    }
                    meta_row.update(dist_only)
                    pair_rows.append(meta_row)
                continue

            # distance cached in pair CSV but missing from global registry
            if (exp_key in done_dist_exp_keys) and not RECOMPUTE_DISTANCES:
                cached_dist_row = get_cached_distance_row(df_dist_pair, exp_key)
                if cached_dist_row is not None:
                    dist_only = {
                        k: v for k, v in cached_dist_row.items()
                        if k not in {"pair", "source", "target", "exp_key", "G", "Y", "seed"}
                    }
                    meta_row.update(dist_only)
                    pair_rows.append(meta_row)
                    done_exp_keys.add(exp_key)
                    continue

            # otherwise compute distance in parallel later
            distance_jobs.append((
                (
                    pair_key,
                    src,
                    tgt,
                    G,
                    Y,
                    seed,
                    df_src,
                    df_holdout,
                    df_ft,
                    FEATURE_COLS,
                    global_scaler,
                    STATIC_COLS,
                    MONTHLY_COLS,
                ),
                meta_row,
            ))

        except Exception as e:
            errs.append({
                "G": G,
                "Y": Y,
                "seed": seed,
                "error": repr(e),
            })

    # parallel distance computation for missing experiments
    if distance_jobs:
        new_dist_rows = []
        meta_by_exp_key = {meta["exp_key"]: meta for _, meta in distance_jobs}

        with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
            futures = {
                ex.submit(compute_one_distance_task, args): meta["exp_key"]
                for args, meta in distance_jobs
            }

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{pair_key} distance jobs",
                leave=False,
            ):
                exp_key = futures[fut]
                try:
                    dist_row = fut.result()
                    new_dist_rows.append(dist_row)

                    meta_row = meta_by_exp_key[exp_key].copy()
                    dist_only = {
                        k: v for k, v in dist_row.items()
                        if k not in {"pair", "source", "target", "exp_key", "G", "Y", "seed"}
                    }
                    meta_row.update(dist_only)
                    pair_rows.append(meta_row)
                    done_exp_keys.add(exp_key)
                    done_dist_exp_keys.add(exp_key)

                except Exception as e:
                    # recover G/Y/seed from meta row for easier debugging
                    meta = meta_by_exp_key[exp_key]
                    errs.append({
                        "G": meta["G"],
                        "Y": meta["Y"],
                        "seed": meta["seed"],
                        "error": repr(e),
                    })

        if new_dist_rows:
            df_dist_pair = pd.concat(
                [df_dist_pair, pd.DataFrame(new_dist_rows)],
                axis=0,
                ignore_index=True,
            )

    if skipped:
        skipped_tasks_by_pair[pair_key] = skipped
    if errs:
        errors_assets_by_pair[pair_key] = errs

    # save pair distance registry once per pair
    df_dist_pair = df_dist_pair.drop_duplicates(subset=["exp_key"], keep="last")
    df_dist_pair.to_csv(dist_registry_path, index=False)

    # merge pair rows into global experiment registry
    experiment_rows.extend(pair_rows)

    # save global registry incrementally
    df_tmp = pd.DataFrame(experiment_rows)
    df_tmp = df_tmp.drop_duplicates(subset=["exp_key"], keep="last")
    df_tmp.to_csv(registry_path, index=False)

    print(
        f"\n{pair_key}: total_registry_rows={len(df_tmp)} | "
        f"pair_dist_rows={len(df_dist_pair)} | "
        f"new_pair_rows={len(pair_rows)} | "
        f"distance_jobs={len(distance_jobs)} | "
        f"skipped={len(skipped)} | errors={len(errs)} | "
        f"G_max={G_max}, Y_max={Y_max}"
    )

df_experiments = pd.DataFrame(experiment_rows)
df_experiments = df_experiments.drop_duplicates(subset=["exp_key"], keep="last")
df_experiments.to_csv(registry_path, index=False)

print("Total experiments:", len(df_experiments))

In [ ]:
registry_path = RESULTS_DIR / "experiment_registry_all_pairs.csv"

if not registry_path.exists():
    raise FileNotFoundError(f"Missing registry: {registry_path}")

df_experiments = pd.read_csv(registry_path)
print(f"Loaded {len(df_experiments)} experiments from {registry_path}")
display(df_experiments.head())

### Run TL datasets & models:

### Compute E_ZERO:
E_ZERO = error of the CH baseline model evaluated on the fixed ISL holdout dataset (unseen glaciers), with no finetuning

In [ ]:
RUN_E_ZERO = True

if RUN_E_ZERO:
    E0_rows = []
    MAKE_PLOTS = False

    for pair_key in tqdm(sorted(static_assets_by_pair.keys()),
                         desc="E_ZERO for all directed pairs"):
        src, tgt = pair_key.split("_to_")

        if src not in baseline_models:
            E0_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing baseline_model",
            })
            continue

        static_assets = static_assets_by_pair[pair_key]
        model_src = baseline_models[src]

        ax = None
        fig = None
        if MAKE_PLOTS:
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))

        try:
            metrics_zero, df_preds_zero, _, _ = evaluate_one_model_TL(
                cfg=cfg,
                model=model_src,
                device=device,
                tl_assets_for_key=static_assets,
                ax=ax,
                title=f"E_ZERO: {src} baseline on {tgt} holdout",
                batch_size=128,
                domain_vocab=static_assets.get("domain_vocab", None),
                show_plot=MAKE_PLOTS,
            )

            E_ZERO_a = float(metrics_zero["RMSE_annual"])
            E_ZERO_w = float(metrics_zero["RMSE_winter"])
            E_ZERO = float(np.mean([E_ZERO_a, E_ZERO_w]))

            row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "E_ZERO_a": E_ZERO_a,
                "E_ZERO_w": E_ZERO_w,
                "E_ZERO": E_ZERO,
            }

            for k in ["R2_annual", "R2_winter", "bias_annual", "bias_winter"]:
                if k in metrics_zero:
                    row[k] = metrics_zero[k]

            E0_rows.append(row)

            print(f"{pair_key}: E0_a={E_ZERO_a:.3f} | "
                  f"E0_w={E_ZERO_w:.3f} | E0={E_ZERO:.3f}")

            if MAKE_PLOTS:
                plt.show()

            del df_preds_zero

        except Exception as e:
            if MAKE_PLOTS and fig is not None:
                plt.close(fig)

            E0_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": repr(e),
            })

    df_E0 = pd.DataFrame(E0_rows).sort_values(["source", "target"])
    df_E0.to_csv(RESULTS_DIR / "df_E0_all_pairs.csv", index=False)

else:
    df_E0 = pd.read_csv(RESULTS_DIR / "df_E0_all_pairs.csv")

display(df_E0)

### E_TL:

#### Train adapter-only models for experiments:

In [ ]:
# load GS results:
with open(
        BASE_DIR / "tuning_adapter/adapter_best_by_region_2026-02-24_15.json",
        "r") as f:
    best_by_region = json.load(f)

In [ ]:
def get_one_tl_asset_from_registry_row(
    *,
    cfg,
    row,
    pair_splits,
    static_assets_by_pair,
    all_pair_res,
    MONTHLY_COLS,
    STATIC_COLS,
):
    """
    Reconstruct one TL asset dict for one experiment row.
    Uses deterministic resampling + build_or_load cache behavior.
    """
    pair_key = row["pair"]
    exp_key = row["exp_key"]
    seed = int(row["seed"])
    G = int(row["G"])
    Y = int(row["Y"])

    d = pair_splits[pair_key]
    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    pool_glaciers = d["pool_glaciers"]
    df_pool_rows = d["df_target"][d["df_target"]["GLACIER"].isin(
        pool_glaciers)].copy()

    df_ft, chosen, _ = sample_monitoring_subset_from_pool(
        df_pool=df_pool_rows,
        G=G,
        Y=Y,
        seed=seed,
        glacier_pick_method="random",
        year_pick_method="earliest_block",
        enforce_full_Y_if_possible=True,
    )

    cache_dir_pair = Path(row["cache_dir"])

    assets_dict = build_budget_assets_finetune_only(
        cfg=cfg,
        res_xreg=res_xreg,
        static_assets=static_assets,
        df_ft=df_ft,
        exp_key=exp_key,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=cache_dir_pair,
        force_recompute=False,
        val_ratio=0.2,
        show_progress=False,
        seed=seed,
    )

    return assets_dict[exp_key]

In [ ]:
RUN_EVAL = True
RELOAD_EXISTING_EVAL = True
SAVE_EVERY = 1  # safest; can set to 5 or 10 later

results_eval_path = RESULTS_DIR / "df_etl_adapter_all.csv"
infos_eval_path = RESULTS_DIR / "df_tl_infos.csv"
errors_eval_path = RESULTS_DIR / "df_tl_errors.csv"

# pair_key -> (E0_a, E0_w)
E0_lookup = (
    df_E0.set_index("pair")[["E_ZERO_a", "E_ZERO_w"]]
    .to_dict(orient="index")
)

# --------------------------------------------------
# Load existing saved outputs if present
# --------------------------------------------------
if RELOAD_EXISTING_EVAL and results_eval_path.exists():
    df_results_existing = pd.read_csv(results_eval_path)
    results_rows = df_results_existing.to_dict(orient="records")
    done_result_exp_keys = set(df_results_existing["exp_key"].astype(str))
    print(f"Loaded existing eval results for {len(done_result_exp_keys)} experiments")
else:
    results_rows = []
    done_result_exp_keys = set()

if RELOAD_EXISTING_EVAL and infos_eval_path.exists():
    df_infos_existing = pd.read_csv(infos_eval_path)
    infos_tl_rows = df_infos_existing.to_dict(orient="records")
    done_info_run_keys = set(df_infos_existing["run_key"].astype(str))
else:
    infos_tl_rows = []
    done_info_run_keys = set()

if RELOAD_EXISTING_EVAL and errors_eval_path.exists():
    df_errors_existing = pd.read_csv(errors_eval_path)
    errors_tl_rows = df_errors_existing.to_dict(orient="records")
else:
    errors_tl_rows = []

# optional: keep preds only if really needed
# preds_tl_rows = []

n_since_save = 0

for row in tqdm(
    df_experiments.sort_values(["pair", "G", "Y", "seed"]).to_dict(orient="records"),
    desc="Train + evaluate TL experiments",
):
    pair_key = row["pair"]
    src = row["source"]
    tgt = row["target"]
    exp_key = row["exp_key"]
    run_key = f"{exp_key}__adapter"

    # if already evaluated successfully, skip
    if RUN_EVAL and exp_key in done_result_exp_keys:
        continue

    try:
        # --------------------------------------------------
        # 1) rebuild/load ONE asset
        # --------------------------------------------------
        assets = get_one_tl_asset_from_registry_row(
            cfg=cfg,
            row=row,
            pair_splits=pair_splits,
            static_assets_by_pair=static_assets_by_pair,
            all_pair_res=all_pair_res,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
        )

        # --------------------------------------------------
        # 2) pair-specific model directory
        # --------------------------------------------------
        d = pair_splits[pair_key]
        sig = split_signature(d["holdout_glaciers"])
        model_dir_pair = MODELS_DIR / pair_key / f"holdout_{sig}"
        model_dir_pair.mkdir(parents=True, exist_ok=True)

        # --------------------------------------------------
        # 3) source-specific baseline ckpt + params
        # --------------------------------------------------
        pretrained_ckpt_path = baseline_infos[src]["path"]
        best_params_src = params_by_key[REGION_CODE_MAP[src]]

        # --------------------------------------------------
        # 4) finetune/load ONE model
        # --------------------------------------------------
        model, out_path, info = finetune_or_load_one_TL(
            cfg=cfg,
            exp_key=exp_key,
            tl_assets_for_key=assets,
            best_params=best_params_src,
            device=device,
            pretrained_ckpt_path=pretrained_ckpt_path,
            models_dir=model_dir_pair,
            prefix=f"lstm_TL_{src}_to_{tgt}",
            strategy="adapter",
            force_retrain=False,
            verbose=False,
            best_by_region=best_by_region,
            date=None,
            load_latest=True,
            skip_if_missing=False,
            prefer_tuned_ckpt=False,
        )

        # save info row if not already present
        if run_key not in done_info_run_keys:
            info_row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key,
                "run_key": run_key,
                "out_path": str(out_path),
                **({k: v for k, v in info.items()} if isinstance(info, dict) else {"info": info}),
            }
            infos_tl_rows.append(info_row)
            done_info_run_keys.add(run_key)

        if not RUN_EVAL:
            del assets, model, out_path, info
            continue

        # --------------------------------------------------
        # 5) evaluate immediately
        # --------------------------------------------------
        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=model,
            device=device,
            tl_assets_for_key=assets,
            ax=None,
            title=None,
            batch_size=128,
            domain_vocab=assets.get("domain_vocab", None),
            show_plot=False,
        )

        E0_a = E0_lookup.get(pair_key, {}).get("E_ZERO_a", float("nan"))
        E0_w = E0_lookup.get(pair_key, {}).get("E_ZERO_w", float("nan"))

        result_row = {
            **row,        # all registry metadata + distances
            **metrics,    # RMSE_annual, RMSE_winter, etc.
            "run_key": run_key,
            "pair": pair_key,
            "source": src,
            "target": tgt,
            "E_ZERO_RMSE_annual": E0_a,
            "Delta_vs_ZERO_annual": metrics["RMSE_annual"] - E0_a,
            "E_ZERO_RMSE_winter": E0_w,
            "Delta_vs_ZERO_winter": metrics["RMSE_winter"] - E0_w,
        }
        results_rows.append(result_row)
        done_result_exp_keys.add(exp_key)

        # optional:
        # if df_preds is not None:
        #     df_preds = df_preds.copy()
        #     df_preds["exp_key"] = exp_key
        #     df_preds["pair"] = pair_key
        #     preds_tl_rows.append(df_preds)

        # --------------------------------------------------
        # 6) explicitly drop heavy objects
        # --------------------------------------------------
        del assets, model, out_path, info, df_preds

    except Exception as e:
        errors_tl_rows.append({
            **row,
            "run_key": run_key,
            "error": repr(e),
        })

    # --------------------------------------------------
    # 7) incremental save
    # --------------------------------------------------
    n_since_save += 1
    if n_since_save >= SAVE_EVERY:
        if infos_tl_rows:
            df_tmp_infos = pd.DataFrame(infos_tl_rows)
            df_tmp_infos = df_tmp_infos.drop_duplicates(subset=["run_key"], keep="last")
            df_tmp_infos.to_csv(infos_eval_path, index=False)

        if errors_tl_rows:
            df_tmp_errors = pd.DataFrame(errors_tl_rows)
            # keep all errors; or deduplicate if you prefer
            df_tmp_errors.to_csv(errors_eval_path, index=False)

        if RUN_EVAL and results_rows:
            df_tmp_results = pd.DataFrame(results_rows)
            df_tmp_results = df_tmp_results.drop_duplicates(subset=["exp_key"], keep="last")
            df_tmp_results.to_csv(results_eval_path, index=False)

        n_since_save = 0

print("\nDone finetuning/evaluation.")

# final save
if infos_tl_rows:
    df_tl_infos = pd.DataFrame(infos_tl_rows).drop_duplicates(subset=["run_key"], keep="last")
    df_tl_infos.to_csv(infos_eval_path, index=False)
else:
    df_tl_infos = pd.DataFrame()

if errors_tl_rows:
    df_tl_errors = pd.DataFrame(errors_tl_rows)
    df_tl_errors.to_csv(errors_eval_path, index=False)
else:
    df_tl_errors = pd.DataFrame()

if RUN_EVAL and results_rows:
    df_etl_adapter_all = (
        pd.DataFrame(results_rows)
        .drop_duplicates(subset=["exp_key"], keep="last")
        .set_index("exp_key")
        .sort_index()
    )
    df_etl_adapter_all.to_csv(results_eval_path)
else:
    df_etl_adapter_all = pd.DataFrame()

display(df_etl_adapter_all.head())

### Compute E_FULL: 
Adapter fine-tuned on all ISL pool data (everything that is not in the fixed holdout glaciers), evaluated on the same fixed holdout (ds_test).

In [ ]:
from pathlib import Path

def get_one_fullpool_asset(
    *,
    cfg,
    pair_key,
    pair_splits,
    static_assets_by_pair,
    all_pair_res,
    MONTHLY_COLS,
    STATIC_COLS,
):
    d = pair_splits[pair_key]
    src = d["source"]
    tgt = d["target"]

    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    pool_glaciers = d["pool_glaciers"]
    df_pool_rows = d["df_target"][d["df_target"]["GLACIER"].isin(pool_glaciers)].copy()

    if len(df_pool_rows) == 0:
        raise ValueError("empty df_pool_rows")

    sig = split_signature(d["holdout_glaciers"])
    cache_dir_pair = Path(CACHE_DIR) / pair_key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    exp_key_full = f"TL_{src}_to_{tgt}_FULLPOOL"

    assets_dict = build_budget_assets_finetune_only(
        cfg=cfg,
        res_xreg=res_xreg,
        static_assets=static_assets,
        df_ft=df_pool_rows,
        exp_key=exp_key_full,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=cache_dir_pair,
        force_recompute=False,
        val_ratio=0.2,
        show_progress=False,
        seed=cfg.seed,
    )

    return exp_key_full, assets_dict[exp_key_full], df_pool_rows

In [ ]:
RUN_E_FULL = True

if RUN_E_FULL:
    EFULL_rows = []
    infos_full_rows = []
    errors_full_rows = []

    for pair_key in tqdm(sorted(pair_splits.keys()), desc="FULLPOOL train + eval"):
        d = pair_splits[pair_key]

        if "error" in d:
            errors_full_rows.append({
                "pair": pair_key,
                "source": d.get("source"),
                "target": d.get("target"),
                "error": d["error"],
            })
            continue

        src = d["source"]
        tgt = d["target"]

        if pair_key not in static_assets_by_pair:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing static_assets_by_pair",
            })
            continue

        if src not in baseline_infos:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing baseline_infos",
            })
            continue

        try:
            # 1) build/load one FULLPOOL asset
            exp_key_full, assets_full, df_pool_rows = get_one_fullpool_asset(
                cfg=cfg,
                pair_key=pair_key,
                pair_splits=pair_splits,
                static_assets_by_pair=static_assets_by_pair,
                all_pair_res=all_pair_res,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
            )

            # 2) model dir
            sig = split_signature(d["holdout_glaciers"])
            model_dir_pair = MODELS_DIR / pair_key / f"holdout_{sig}"
            model_dir_pair.mkdir(parents=True, exist_ok=True)

            # 3) source-specific baseline ckpt + params
            pretrained_ckpt_path = baseline_infos[src]["path"]
            best_params_src = params_by_key[REGION_CODE_MAP[src]]

            # 4) finetune/load one FULLPOOL model
            model_full, out_path_full, info_full = finetune_or_load_one_TL(
                cfg=cfg,
                exp_key=exp_key_full,
                tl_assets_for_key=assets_full,
                best_params=best_params_src,
                device=device,
                pretrained_ckpt_path=pretrained_ckpt_path,
                models_dir=model_dir_pair,
                prefix=f"lstm_TL_{src}_to_{tgt}",
                strategy="adapter",
                force_retrain=False,
                verbose=False,
                best_by_region=best_by_region,
                date=None,
                load_latest=True,
                skip_if_missing=False,
                prefer_tuned_ckpt=False,
            )

            infos_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key_full,
                "run_key": f"{exp_key_full}__adapter",
                "out_path": str(out_path_full),
                **({k: v for k, v in info_full.items()} if isinstance(info_full, dict) else {"info": info_full}),
            })

            # 5) evaluate immediately
            metrics_full, df_preds_full, _, _ = evaluate_one_model_TL(
                cfg=cfg,
                model=model_full,
                device=device,
                tl_assets_for_key=assets_full,
                ax=None,
                title=None,
                batch_size=128,
                domain_vocab=assets_full.get("domain_vocab", None),
                show_plot=False,
            )

            E_FULL_a = float(metrics_full["RMSE_annual"])
            E_FULL_w = float(metrics_full["RMSE_winter"])
            E_FULL = float(np.mean([E_FULL_a, E_FULL_w]))

            row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key_full,
                "run_key": f"{exp_key_full}__adapter",
                "n_rows_ft": int(len(df_pool_rows)),
                "n_glaciers_ft": int(df_pool_rows["GLACIER"].nunique()),
                "n_years_ft": int(df_pool_rows["YEAR"].nunique()),
                "E_FULL_a": E_FULL_a,
                "E_FULL_w": E_FULL_w,
                "E_FULL": E_FULL,
            }

            for k in ["R2_annual", "R2_winter", "bias_annual", "bias_winter"]:
                if k in metrics_full:
                    row[k] = metrics_full[k]

            EFULL_rows.append(row)

            print(
                f"{pair_key}: E_FULL_a={E_FULL_a:.3f} | "
                f"E_FULL_w={E_FULL_w:.3f} | E_FULL={E_FULL:.3f}"
            )

            del assets_full, model_full, out_path_full, info_full, df_preds_full

        except Exception as e:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": repr(e),
            })

    df_EFULL = pd.DataFrame(EFULL_rows).sort_values(["source", "target"])
    df_EFULL.to_csv(RESULTS_DIR / "df_EFULL_all_pairs.csv", index=False)

    df_EFULL_infos = pd.DataFrame(infos_full_rows)
    df_EFULL_errors = pd.DataFrame(errors_full_rows)

else:
    df_EFULL = pd.read_csv(RESULTS_DIR / "df_EFULL_all_pairs.csv")

display(df_EFULL)

### E_scratch:

In [ ]:
RUN_SCRATCH = False

if RUN_SCRATCH:
    _re_budget = re.compile(r"_G(?P<G>\d+)_Y(?P<Y>\d+)_seed(?P<seed>\d+)")

    def parse_budget(exp_key: str):
        m = _re_budget.search(str(exp_key))
        if not m:
            return {"G": np.nan, "Y": np.nan, "seed": np.nan}
        return {k: int(v) for k, v in m.groupdict().items()}

    def within_assets_from_tl_assets(tl_assets_for_key: dict):
        """
        Train only on the fine-tuning subset and evaluate on the holdout test set.
        """
        return {
            "ds_train": tl_assets_for_key["ds_finetune"],
            "ds_test": tl_assets_for_key["ds_test"],
            "train_idx": tl_assets_for_key["finetune_train_idx"],
            "val_idx": tl_assets_for_key["finetune_val_idx"],
        }

    def infer_src_tgt_from_exp_key(exp_key: str):
        m = re.search(r"TL_(?P<src>.+?)_to_(?P<tgt>.+?)_G\d+_Y\d+_seed\d+",
                      str(exp_key))
        if not m:
            return None, None
        return m.group("src"), m.group("tgt")

    models_within = {}
    infos_within = {}
    rows = []

    exp_keys = sorted(assets_all.keys())
    pbar = tqdm(
        exp_keys,
        desc="Training + evaluating scratch models",
        dynamic_ncols=True,
    )

    for exp_key in pbar:
        meta = parse_budget(exp_key)
        if not np.isnan(meta["G"]):
            pbar.set_postfix(
                G=int(meta["G"]),
                Y=int(meta["Y"]),
                seed=int(meta["seed"]),
            )

        tl_assets_for_key = assets_all[exp_key]
        w_assets = within_assets_from_tl_assets(tl_assets_for_key)

        # Prefer metadata from assets if available
        src = tl_assets_for_key.get("source_code", None)
        tgt = tl_assets_for_key.get("target_code", None)

        if src is None or tgt is None:
            src2, tgt2 = infer_src_tgt_from_exp_key(exp_key)
            src = src if src is not None else src2
            tgt = tgt if tgt is not None else tgt2

        if src is None:
            raise ValueError(
                f"Could not infer source region for exp_key={exp_key}")

        best_params = params_by_key[REGION_CODE_MAP[src]]

        model_w, path_w, info_w = train_or_load_one_within_region_monitoring(
            cfg=cfg,
            key=exp_key,
            lstm_assets=w_assets,
            best_params=best_params,
            device=device,
            models_dir=MODELS_DIR,
            prefix="lstm_within",
            train_flag=True,
            force_retrain=False,
            epochs=150,
            batch_size_train=64,
            batch_size_val=128,
            batch_size_test=128,
            date=None,
            verbose=False,
        )

        models_within[exp_key] = model_w
        infos_within[exp_key] = {"model_path": path_w, **(info_w or {})}

        # Evaluate on holdout
        met_w, df_w = model_w.evaluate_with_preds(
            device,
            info_w["test_dl"],
            info_w["ds_test"],
        )

        scores_annual, scores_winter = compute_seasonal_scores(
            df_w,
            target_col="target",
            pred_col="pred",
        )

        rows.append({
            "exp_key": exp_key,
            "source": src,
            "target": tgt,
            "RMSE_SCRATCH_annual": float(scores_annual["rmse"]),
            "RMSE_SCRATCH_winter": float(scores_winter["rmse"]),
            "R2_SCRATCH_annual": float(scores_annual["R2"]),
            "R2_SCRATCH_winter": float(scores_winter["R2"]),
            "Bias_SCRATCH_annual": float(scores_annual["Bias"]),
            "Bias_SCRATCH_winter": float(scores_winter["Bias"]),
            "n_annual": int(scores_annual.get("n", np.nan)),
            "n_winter": int(scores_winter.get("n", np.nan)),
            **meta,
            "model_path": path_w,
        })

    df_scratch = pd.DataFrame(rows).set_index("exp_key").sort_index()

    final_results_csv = RESULTS_DIR / "df_ESCRATCH_all_pairs.csv"
    df_scratch.to_csv(final_results_csv)

    display(df_scratch.head())

else:
    df_scratch = pd.read_csv(final_results_csv)

## Evaluate experiments:

### Build one unified results dataframe (with G/Y/M/seed parsed):

In [ ]:
# ----------------------------
# 1) Parse G/Y/seed from exp_key
# ----------------------------
_re_budget = re.compile(r"_G(?P<G>\d+)_Y(?P<Y>\d+)_seed(?P<seed>\d+)")


def parse_budget(s: str):
    m = _re_budget.search(str(s))
    if not m:
        return {"G": np.nan, "Y": np.nan, "seed": np.nan}
    return {k: int(v) for k, v in m.groupdict().items()}


# ----------------------------
# 2) Start from TL experiment results
# ----------------------------
df = df_etl_adapter_all.copy()

if "exp_key" not in df.columns:
    df = df.reset_index()

# ----------------------------
# 3) Add parsed budget columns
# ----------------------------
meta = df["exp_key"].apply(parse_budget).apply(pd.Series)
df = pd.concat([df, meta], axis=1)

# Drop non-budget rows (e.g. FULLPOOL if present)
df = df.dropna(subset=["G", "Y", "seed"]).copy()

df["G"] = df["G"].astype(int)
df["Y"] = df["Y"].astype(int)
df["seed"] = df["seed"].astype(int)

# ----------------------------
# 4) Merge pair-specific E_ZERO
# ----------------------------
df = df.merge(
    df_E0[["pair", "E_ZERO_a", "E_ZERO_w", "E_ZERO"]],
    on="pair",
    how="left",
)

# ----------------------------
# 5) Merge pair-specific E_FULL
# ----------------------------
df = df.merge(
    df_EFULL[["pair", "E_FULL_a", "E_FULL_w", "E_FULL"]],
    on="pair",
    how="left",
)

# ----------------------------
# 6) Build a flat dataframe of pair-level topoclimatic distances
# ----------------------------
pair_distance_rows = []
for pair_key, d in pair_splits.items():
    if "error" in d:
        continue

    row = {
        "pair": pair_key,
        "source": d["source"],
        "target": d["target"],
    }
    row.update(d["distances"])
    pair_distance_rows.append(row)

df_pair_dist = pd.DataFrame(pair_distance_rows)

# Merge pair-level distances
df = df.merge(
    df_pair_dist,
    on=["pair", "source", "target"],
    how="left",
)

# ----------------------------
# 7) Merge experiment-level src->ft distances
# ----------------------------
df = df.merge(
    df_dist_src_ft[[
        "exp_key",
        "n_ft",
        "D_wass_src_ft",
        "D_energy_src_ft",
        "D_wass_ft_holdout",
        "D_energy_ft_holdout",
    ]],
    on="exp_key",
    how="left",
)

# # ----------------------------
# # 7b) Merge scratch-training results (E_SCRATCH)
# # ----------------------------
# if df_scratch is not None:

#     df_scratch_reset = df_scratch.reset_index()

#     df = df.merge(
#         df_scratch_reset[[
#             "exp_key",
#             "RMSE_SCRATCH_annual",
#             "RMSE_SCRATCH_winter",
#             "R2_SCRATCH_annual",
#             "R2_SCRATCH_winter",
#             "Bias_SCRATCH_annual",
#             "Bias_SCRATCH_winter",
#         ]],
#         on="exp_key",
#         how="left",
#     )

#     # mean scratch RMSE
#     df["RMSE_SCRATCH"] = 0.5 * (df["RMSE_SCRATCH_annual"] +
#                                 df["RMSE_SCRATCH_winter"])

# ----------------------------
# 8) Derived quantities
# ----------------------------
df["Effort"] = df["G"] * df["Y"]
df["RMSE_TL"] = 0.5 * (df["RMSE_annual"] + df["RMSE_winter"])

# Deltas vs ZERO
df["Delta_vs_ZERO_annual"] = df["RMSE_annual"] - df["E_ZERO_a"]
df["Delta_vs_ZERO_winter"] = df["RMSE_winter"] - df["E_ZERO_w"]
df["Delta_vs_ZERO"] = df["RMSE_TL"] - df["E_ZERO"]

df["GapClosed_annual"] = ((df["E_ZERO_a"] - df["RMSE_annual"]) /
                          (df["E_ZERO_a"] - df["E_FULL_a"]))

df["GapClosed_winter"] = ((df["E_ZERO_w"] - df["RMSE_winter"]) /
                          (df["E_ZERO_w"] - df["E_FULL_w"]))

df["GapClosed_mean"] = ((df["E_ZERO"] - df["RMSE_TL"]) /
                        (df["E_ZERO"] - df["E_FULL"]))

df["GapClosed_annual"] = df["GapClosed_annual"].clip(-0.25, 1.25)
df["GapClosed_winter"] = df["GapClosed_winter"].clip(-0.25, 1.25)
df["GapClosed_mean"] = df["GapClosed_mean"].clip(-0.25, 1.25)

# ----------------------------
# 9) Quick sanity checks
# ----------------------------
print("Missing E_ZERO:", df["E_ZERO"].isna().sum())
print("Missing E_FULL:", df["E_FULL"].isna().sum())
print("Missing pair distances:", df["D_energy_src_tgt_all"].isna().sum())
print("Missing FT distances:", df["D_energy_src_ft"].isna().sum())

# Save final merged dataframe for analysis
final_merged_csv = RESULTS_DIR / "df_final_merged_all_pairs.csv"
df.to_csv(final_merged_csv, index=False)
df.head()

### Tier sweep plots (which knob matters most?)

In [ ]:
df_bounds = (df_E0[["pair", "E_ZERO_a", "E_ZERO_w", "E_ZERO"]].merge(
    df_EFULL[["pair", "E_FULL_a", "E_FULL_w", "E_FULL"]],
    on="pair",
    how="inner",
))
bounds_lookup = df_bounds.set_index("pair").to_dict(orient="index")

# pair-level distances
df_dist = (df[[
    "pair", "D_energy_src_tgt_all", "D_wass_src_tgt_all"
]].dropna(subset=["pair"]).drop_duplicates(subset=["pair"]).set_index("pair"))
dist_lookup = df_dist.to_dict(orient="index")

# first keep only valid pairs
pair_keys = [
    pair_key for pair_key in sorted(df_etl_adapter_by_pair.keys())
    if pair_key in bounds_lookup
]

# then sort by energy distance
pair_keys = sorted(
    pair_keys,
    key=lambda p: dist_lookup.get(p, {}).get("D_energy_src_tgt_all", np.inf))

n_pairs = len(pair_keys)

fig, axes = plt.subplots(
    n_pairs,
    2,
    figsize=(16, 4 * n_pairs),
    sharex=False,
    sharey=False,
    squeeze=False,
)

for i, pair_key in enumerate(pair_keys):
    df_pair = df_etl_adapter_by_pair[pair_key].copy()
    src, tgt = pair_key.split("_to_")
    b = bounds_lookup[pair_key]
    d = dist_lookup.get(pair_key, {})

    meta = df_pair.index.to_series().apply(parse_budget).apply(pd.Series)
    df_plot = pd.concat([meta, df_pair], axis=1)
    df_plot = df_plot.dropna(subset=["G", "Y", "seed"]).copy()

    if len(df_plot) == 0:
        axes[i, 0].set_visible(False)
        axes[i, 1].set_visible(False)
        continue

    df_plot["G"] = df_plot["G"].astype(int)
    df_plot["Y"] = df_plot["Y"].astype(int)
    df_plot["seed"] = df_plot["seed"].astype(int)

    show_legend = (i == 5)

    plot_sweep_Y_by_G_colormap(
        df_plot,
        b["E_ZERO_a"],
        b["E_FULL_a"],
        b["E_ZERO_w"],
        b["E_FULL_w"],
        b["E_ZERO"],
        b["E_FULL"],
        metric="annual",
        title=f"{src} → {tgt} | annual",
        ax=axes[i, 0],
        show_legend=False,
    )

    plot_sweep_Y_by_G_colormap(
        df_plot,
        b["E_ZERO_a"],
        b["E_FULL_a"],
        b["E_ZERO_w"],
        b["E_FULL_w"],
        b["E_ZERO"],
        b["E_FULL"],
        metric="winter",
        title=f"{src} → {tgt} | winter",
        ax=axes[i, 1],
        show_legend=show_legend,
    )

    txt = (rf"$D_{{E}}(S,T) = {d.get('D_energy_src_tgt_all', np.nan):.2f}$"
           "\n"
           rf"$D_{{W}}(S,T) = {d.get('D_wass_src_tgt_all', np.nan):.2f}$")

    for j in [0, 1]:
        axes[i, j].text(
            0.02,
            0.98,
            txt,
            transform=axes[i, j].transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.8,
                edgecolor="none",
            ),
        )

plt.tight_layout()
plt.show()

### Contour distance plots:

In [ ]:
def plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_annual",
    dist_col="D_energy_src_tgt_all",
    g_col="G",
    y_col="Y",
    pair_col="pair",
    q=3,
    gmin=1,
    ymin=1,
    bin_labels=None,
    levels_filled=None,
    levels_lines=None,
    cmap="viridis",
    vmin=0.0,
    vmax=1.0,
    figsize=None,
    sharey=True,
    show_pair_labels=True,
    show_bounds_in_title=True,
    show_n_points_in_title=True,
    pair_label_transform=lambda p: p.replace("_to_", " → "),
    xlabel="Years monitored (Y)",
    ylabel="Glaciers monitored (G)",
    suptitle="Monitoring recovery by topoclimatic transfer distance",
    cbar_label=None,
    title_dist_symbol=r"D_E^{src\to tgt}",
    scatter_points=True,
    scatter_kwargs=None,
    text_kwargs=None,
    contour_kwargs=None,
    contourf_kwargs=None,
):
    """
    Plot mean recovery surfaces over (G, Y), grouped by quantile bins of a distance metric.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing metric, distance, G, Y, and pair columns.
    metric_col : str
        Column to plot as the surface value.
    dist_col : str
        Distance column used for binning.
    g_col : str
        Column for number of glaciers monitored.
    y_col : str
        Column for number of years monitored.
    pair_col : str
        Column identifying transfer pairs.
    q : int
        Number of quantile bins.
    bin_labels : list[str] or None
        Labels for the distance bins. If None, defaults depend on q.
    levels_filled : array-like or None
        Filled contour levels. If None, uses np.linspace(vmin, vmax, 11).
    levels_lines : list[float] or None
        Contour line levels. If None, uses [0.25, 0.50, 0.75, 1.00].
    cmap : str
        Colormap for filled contours.
    vmin, vmax : float
        Color scale limits.
    figsize : tuple or None
        Figure size. If None, uses (6*q, 5).
    sharey : bool
        Whether subplots share the y-axis.
    show_pair_labels : bool
        Whether to annotate each panel with the pairs in that bin.
    show_bounds_in_title : bool
        Whether to include distance bin bounds in each panel title.
    show_n_points_in_title : bool
        Whether to include the number of rows in the bin title.
    pair_label_transform : callable
        Function to convert raw pair keys to display labels.
    xlabel, ylabel : str
        Axis labels.
    suptitle : str
        Figure super-title.
    cbar_label : str or None
        Colorbar label. If None, uses metric_col.
    title_dist_symbol : str
        MathText string for the distance symbol in titles.
    scatter_points : bool
        Whether to overlay the actual (Y, G) grid points.
    scatter_kwargs, text_kwargs, contour_kwargs, contourf_kwargs : dict or None
        Optional matplotlib styling dictionaries.

    Returns
    -------
    fig, axes : matplotlib Figure and Axes
    """
    # Filter to min G and Y
    df = df[df[g_col] >= gmin].copy()
    df = df[df[y_col] >= ymin].copy()

    required_cols = [metric_col, dist_col, g_col, y_col, pair_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df_plot = df.dropna(subset=required_cols).copy()
    if df_plot.empty:
        raise ValueError("No valid rows left after dropping NaNs.")

    if bin_labels is None:
        default_labels = {
            2: [
                "Low distance (source ≈ fine-tune target)",
                "High distance (strong domain shift)"
            ],
            3: [
                "Low distance (source ≈ fine-tune target)",
                "Medium distance (moderate domain shift)",
                "High distance (strong domain shift)"
            ],
            4: ["Very low", "Low", "High", "Very high"],
        }
        bin_labels = default_labels.get(q, [f"Bin {i+1}" for i in range(q)])

    if len(bin_labels) != q:
        raise ValueError(
            f"bin_labels must have length {q}, got {len(bin_labels)}.")

    if levels_filled is None:
        levels_filled = np.linspace(vmin, vmax, 11)
    if levels_lines is None:
        levels_lines = [0.25, 0.50, 0.75, 1.00]
    if figsize is None:
        figsize = (6 * q, 5)
    if cbar_label is None:
        cbar_label = metric_col

    if scatter_kwargs is None:
        scatter_kwargs = dict(s=16, alpha=0.45)
    if text_kwargs is None:
        text_kwargs = dict(
            x=0.03,
            y=0.97,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.85,
                edgecolor="none",
            ),
        )
    if contour_kwargs is None:
        contour_kwargs = dict(colors="k", linewidths=1.0)
    if contourf_kwargs is None:
        contourf_kwargs = {}

    # Keep raw interval bins
    df_plot["dist_bin_raw"] = pd.qcut(df_plot[dist_col], q=q)

    raw_bins = df_plot["dist_bin_raw"].cat.categories
    bin_label_map = {
        raw_bin: label
        for raw_bin, label in zip(raw_bins, bin_labels)
    }
    df_plot["dist_bin"] = df_plot["dist_bin_raw"].map(bin_label_map)

    pair_bin = (df_plot[[pair_col, "dist_bin"]].drop_duplicates().sort_values(
        ["dist_bin", pair_col]))

    fig, axes = plt.subplots(1,
                             q,
                             figsize=figsize,
                             sharey=sharey,
                             squeeze=False)
    axes = axes.ravel()

    im = None

    for ax, raw_bin, label in zip(axes, raw_bins, bin_labels):
        sub = df_plot[df_plot["dist_bin_raw"] == raw_bin].copy()
        n_points = len(sub)

        heat = (sub.groupby([g_col, y_col
                             ])[metric_col].mean().unstack(y_col).sort_index(
                                 axis=0).sort_index(axis=1))

        if heat.empty:
            ax.set_visible(False)
            continue

        y_vals = heat.columns.to_numpy(dtype=float)
        g_vals = heat.index.to_numpy(dtype=float)
        X, Y = np.meshgrid(y_vals, g_vals)
        Z = heat.to_numpy(dtype=float)

        cf = ax.contourf(
            X,
            Y,
            Z,
            levels=levels_filled,
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            **contourf_kwargs,
        )
        im = cf

        # Only draw contour lines when there is enough variation
        zmin = np.nanmin(Z)
        zmax = np.nanmax(Z)
        valid_line_levels = [lv for lv in levels_lines if zmin <= lv <= zmax]
        if len(valid_line_levels) > 0:
            cs = ax.contour(
                X,
                Y,
                Z,
                levels=valid_line_levels,
                **contour_kwargs,
            )
            ax.clabel(cs, fmt="%.2f", inline=True, fontsize=8)

        if scatter_points:
            ax.scatter(X, Y, **scatter_kwargs)

        if show_pair_labels:
            pairs_here = sorted(pair_bin.loc[pair_bin["dist_bin"] == label,
                                             pair_col].tolist())
            pair_txt = "\n".join([pair_label_transform(p) for p in pairs_here])

            if pair_txt.strip():
                ax.text(
                    text_kwargs.pop("x"),
                    text_kwargs.pop("y"),
                    pair_txt,
                    transform=ax.transAxes,
                    **text_kwargs,
                )
                text_kwargs["x"] = 0.03
                text_kwargs["y"] = 0.97

        title_lines = [label]

        if show_bounds_in_title:
            lo = raw_bin.left
            hi = raw_bin.right
            title_lines.append(
                f"${lo:.2f} < {title_dist_symbol} \\leq {hi:.2f}$")

        if show_n_points_in_title:
            title_lines.append(f"$n={n_points}$")

        ax.set_title("\n".join(title_lines))

        ax.set_xlabel(xlabel)
        ax.set_xticks(y_vals)
        ax.set_yticks(g_vals)

    axes[0].set_ylabel(ylabel)

    fig.suptitle(suptitle, y=1.0)

    plt.tight_layout()
    fig.subplots_adjust(right=0.88, top=0.75)

    cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(cbar_label)

    return fig, axes

#### Energy distance:

##### SRC -> TGT all:

In [ ]:
fig, axes = plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_annual",
    dist_col="D_energy_src_tgt_all",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    suptitle=
    "Monitoring annual PMB recovery by topoclimatic transfer distance (Energy)",
    cbar_label="GapClosed (annual)",
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25],
    levels_lines=[0.25, 0.50, 0.75, 0.8, 1.00],
    title_dist_symbol=r"D_E^{S\to T}",
    gmin=3,
    ymin=3)
plt.show()

##### SRC -> FT Set:

In [ ]:
fig, axes = plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_annual",
    dist_col="D_energy_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    suptitle=
    "Monitoring annual PMB recovery by topoclimatic transfer distance (Energy)",
    cbar_label="GapClosed (annual)",
    title_dist_symbol=r"D_E^{S\to F}",
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25],
    levels_lines=[0.25, 0.3, 0.4, 0.50, 0.6, 0.75, 0.8, 1.00],
    gmin=3,
    ymin=3,
)
plt.show()

In [ ]:
fig, axes = plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_winter",
    dist_col="D_energy_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    suptitle=
    "Monitoring winter PMB recovery by topoclimatic transfer distance (Energy)",
    cbar_label="GapClosed (winter)",
    title_dist_symbol=r"D_E^{S\to F}",
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25],
    levels_lines=[0.25, 0.3, 0.4, 0.50, 0.6, 0.75, 0.8, 1.00],
    gmin=3,
    ymin=3)
plt.show()

#### Wasserstein distance:

In [ ]:
fig, axes = plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_annual",
    dist_col="D_wass_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    suptitle=
    "Monitoring annual PMB recovery by topoclimatic transfer distance (Wasserstein)",
    cbar_label="GapClosed (annual)",
    title_dist_symbol=r"D_W^{S\to F}",
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25],
    levels_lines=[0.25, 0.3, 0.4, 0.50, 0.6, 0.75, 0.8, 1.00],
    gmin=3,
    ymin=3)
plt.show()

In [ ]:
fig, axes = plot_recovery_by_distance_bins(
    df,
    metric_col="GapClosed_winter",
    dist_col="D_wass_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    suptitle=
    "Monitoring winter PMB recovery by topoclimatic transfer distance (Wasserstein)",
    cbar_label="GapClosed (winter)",
    title_dist_symbol=r"D_W^{S\to F}",
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.25],
    levels_lines=[0.25, 0.3, 0.4, 0.50, 0.6, 0.75, 0.8, 1.00],
    gmin=3,
    ymin=3)
plt.show()

#### RMSE:

In [ ]:
def plot_metric_by_distance_bins(
    df,
    metric_col="GapClosed_annual",
    dist_col="D_energy_src_tgt_all",
    g_col="G",
    y_col="Y",
    pair_col="pair",
    q=3,
    bin_labels=None,
    levels_filled=None,
    levels_lines=None,
    cmap="viridis",
    vmin=None,
    vmax=None,
    figsize=None,
    sharey=True,
    show_pair_labels=True,
    show_bounds_in_title=True,
    pair_label_transform=lambda p: p.replace("_to_", " → "),
    xlabel="Years monitored (Y)",
    ylabel="Glaciers monitored (G)",
    suptitle="Monitoring response by topoclimatic transfer distance",
    cbar_label=None,
    title_dist_symbol=r"D_E^{src\to tgt}",
    scatter_points=True,
    scatter_kwargs=None,
    text_kwargs=None,
    contour_kwargs=None,
    contourf_kwargs=None,
):
    required_cols = [metric_col, dist_col, g_col, y_col, pair_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df_plot = df.dropna(subset=required_cols).copy()
    if df_plot.empty:
        raise ValueError("No valid rows left after dropping NaNs.")

    if bin_labels is None:
        default_labels = {
            2: [
                "Low distance (source ≈ fine-tune target)",
                "High distance (strong domain shift)"
            ],
            3: [
                "Low distance (source ≈ fine-tune target)",
                "Medium distance (moderate domain shift)",
                "High distance (strong domain shift)"
            ],
            4: ["Very low", "Low", "High", "Very high"],
        }
        bin_labels = default_labels.get(q, [f"Bin {i+1}" for i in range(q)])

    if len(bin_labels) != q:
        raise ValueError(
            f"bin_labels must have length {q}, got {len(bin_labels)}.")

    if vmin is None:
        vmin = float(np.nanmin(df_plot[metric_col]))
    if vmax is None:
        vmax = float(np.nanmax(df_plot[metric_col]))

    if levels_filled is None:
        levels_filled = np.linspace(vmin, vmax, 11)
    if levels_lines is None:
        levels_lines = np.linspace(vmin, vmax, 5)[1:-1]
    if figsize is None:
        figsize = (6 * q, 5)
    if cbar_label is None:
        cbar_label = metric_col

    if scatter_kwargs is None:
        scatter_kwargs = dict(s=16, alpha=0.45)
    if text_kwargs is None:
        text_kwargs = dict(
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.85,
                edgecolor="none",
            ),
        )
    if contour_kwargs is None:
        contour_kwargs = dict(colors="k", linewidths=1.0)
    if contourf_kwargs is None:
        contourf_kwargs = {}

    df_plot["dist_bin_raw"] = pd.qcut(df_plot[dist_col], q=q)

    raw_bins = df_plot["dist_bin_raw"].cat.categories
    bin_label_map = {
        raw_bin: label
        for raw_bin, label in zip(raw_bins, bin_labels)
    }
    df_plot["dist_bin"] = df_plot["dist_bin_raw"].map(bin_label_map)

    pair_bin = (df_plot[[pair_col, "dist_bin"]].drop_duplicates().sort_values(
        ["dist_bin", pair_col]))

    fig, axes = plt.subplots(1,
                             q,
                             figsize=figsize,
                             sharey=sharey,
                             squeeze=False)
    axes = axes.ravel()

    im = None

    for ax, raw_bin, label in zip(axes, raw_bins, bin_labels):
        sub = df_plot[df_plot["dist_bin_raw"] == raw_bin].copy()

        heat = (sub.groupby([g_col, y_col
                             ])[metric_col].mean().unstack(y_col).sort_index(
                                 axis=0).sort_index(axis=1))

        if heat.empty:
            ax.set_visible(False)
            continue

        y_vals = heat.columns.to_numpy(dtype=float)
        g_vals = heat.index.to_numpy(dtype=float)
        X, Y = np.meshgrid(y_vals, g_vals)
        Z = heat.to_numpy(dtype=float)

        cf = ax.contourf(
            X,
            Y,
            Z,
            levels=levels_filled,
            vmin=vmin,
            vmax=vmax,
            cmap=cmap,
            **contourf_kwargs,
        )
        im = cf

        zmin = np.nanmin(Z)
        zmax = np.nanmax(Z)
        valid_line_levels = [lv for lv in levels_lines if zmin <= lv <= zmax]
        if valid_line_levels:
            cs = ax.contour(X,
                            Y,
                            Z,
                            levels=valid_line_levels,
                            **contour_kwargs)
            ax.clabel(cs, fmt="%.2f", inline=True, fontsize=8)

        if scatter_points:
            ax.scatter(X, Y, **scatter_kwargs)

        if show_pair_labels:
            pairs_here = sorted(pair_bin.loc[pair_bin["dist_bin"] == label,
                                             pair_col].tolist())
            pair_txt = "\n".join([pair_label_transform(p) for p in pairs_here])

            if pair_txt.strip():
                ax.text(
                    0.03,
                    0.97,
                    pair_txt,
                    transform=ax.transAxes,
                    **text_kwargs,
                )

        if show_bounds_in_title:
            lo = raw_bin.left
            hi = raw_bin.right
            ax.set_title(
                f"{label}\n${lo:.2f} < {title_dist_symbol} \\leq {hi:.2f}$")
        else:
            ax.set_title(label)

        ax.set_xlabel(xlabel)
        ax.set_xticks(y_vals)
        ax.set_yticks(g_vals)

    axes[0].set_ylabel(ylabel)

    fig.suptitle(suptitle, y=1.0)

    plt.tight_layout()
    fig.subplots_adjust(right=0.88, top=0.82)

    cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label(cbar_label)

    return fig, axes

In [ ]:
df["AbsImprovement_annual"] = df["E_ZERO_a"] - df["RMSE_annual"]
df["AbsImprovement_winter"] = df["E_ZERO_w"] - df["RMSE_winter"]
df["AbsImprovement_mean"] = df["E_ZERO"] - df["RMSE_TL"]

fig, axes = plot_metric_by_distance_bins(
    df,
    metric_col="AbsImprovement_annual",
    dist_col="D_energy_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    levels_filled=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5],
    levels_lines=[0.1, 0.3, 0.5, 0.7, 0.8, 1],
    vmin=0.0,
    vmax=max(1.0, df["AbsImprovement_annual"].max()),
    suptitle=
    "Monitoring annual PMB absolute improvement by topoclimatic transfer distance (Energy)",
    cbar_label="Absolute RMSE improvement (annual)",
    title_dist_symbol=r"D_E^{S\to F}",
)
plt.show()

In [ ]:
dist_col = "D_energy_src_tgt_all"

# one row per pair
df_pairs = (df.groupby("pair", as_index=False).agg({
    dist_col: "first",
    "E_ZERO_a": "first",
    "E_FULL_a": "first"
}))

df_pairs["transfer_gap"] = df_pairs["E_ZERO_a"] - df_pairs["E_FULL_a"]

fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    df_pairs[dist_col],
    df_pairs["transfer_gap"],
    s=80,
)

# label points
for _, r in df_pairs.iterrows():
    ax.text(
        r[dist_col],
        r["transfer_gap"],
        r["pair"].replace("_to_", " → "),
        fontsize=9,
        ha="left",
        va="bottom",
    )

ax.set_xlabel(r"$D_E^{src\to tgt}$")
ax.set_ylabel(r"$E_{ZERO,a} - E_{FULL,a}$")
ax.set_title("Transfer gap vs topoclimatic distance")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plot_metric_by_distance_bins(
    df,
    metric_col="RMSE_annual",
    dist_col="D_energy_src_ft",
    q=3,
    bin_labels=[
        "Low distance (source ≈ fine-tune target)",
        "Medium distance (moderate domain shift)",
        "High distance (strong domain shift)"
    ],
    levels_filled=[0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0],
    levels_lines=[0.4, 0.8, 0.9, 1.0, 1.2, 1.3, 1.4, 1.6, 1.8],
    vmin=0.2,
    vmax=2.0,
    suptitle="Transfer-learning RMSE by monitoring effort and domain distance",
    cbar_label="RMSE (annual)",
    title_dist_symbol=r"D_E^{S\to F}",
)

plt.show()

### Performance vs distance:

In [ ]:
def _make_symmetric_pair_label(pair: str) -> str:
    src, tgt = pair.split("_to_")
    a, b = sorted([src, tgt])
    return f"{a} ↔ {b}"


def plot_distance_vs_metric_by_symmetric_pair(
    df,
    dist_col="D_wass_src_ft",
    metric_col="RMSE_annual",
    group_cols=("G", "Y"),
    pair_col="pair",
    figsize_per_panel=(6, 5),
    xlabel=None,
    ylabel=None,
    title=None,
    annotate=False,
    color_by="pair",  # "pair", "G", "Y", "seed", "D_wass_ft_holdout", or None
    min_g=1,
    min_y=1,
    show_yerr=True,
    show_xerr=False,
    cmap="viridis",
    legend=True,
    symmetric_axes=False,
    aggregate=True,  # NEW: if False, plot all repetitions directly
    alpha=0.85,
    s=60,
):
    """
    Scatter plot of distance vs metric with one panel per symmetric pair.

    Parameters
    ----------
    aggregate : bool
        If True, aggregate over group_cols (and pair direction) and plot mean/std.
        If False, plot all rows (e.g. all seeds / repetitions) directly.
    """
    df = df[df["G"] >= min_g].copy()
    df = df[df["Y"] >= min_y].copy()

    if pair_col not in df.columns:
        raise KeyError(f"'{pair_col}' not found in dataframe.")

    df["pair_sym"] = df[pair_col].apply(_make_symmetric_pair_label)

    cols = [dist_col, metric_col, pair_col, "pair_sym", *group_cols]
    if color_by is not None and color_by not in cols:
        cols.append(color_by)

    df_plot = df.dropna(subset=cols).copy()

    # -----------------------
    # Build plotting table
    # -----------------------
    if aggregate:
        gb_cols = ["pair_sym", pair_col, *group_cols]

        agg_dict = dict(
            x=(dist_col, "mean"),
            x_std=(dist_col, "std"),
            y=(metric_col, "mean"),
            y_std=(metric_col, "std"),
            n=(metric_col, "size"),
        )

        if color_by is not None and color_by not in gb_cols:
            agg_dict["color_val"] = (color_by, "mean")

        plot_df = (df_plot.groupby(gb_cols).agg(**agg_dict).reset_index())
    else:
        plot_df = df_plot.copy()
        plot_df = plot_df.rename(columns={dist_col: "x", metric_col: "y"})
        plot_df["x_std"] = np.nan
        plot_df["y_std"] = np.nan
        plot_df["n"] = 1

        if color_by is not None and color_by not in plot_df.columns:
            raise KeyError(f"`color_by='{color_by}'` not found in dataframe.")

    sym_pairs = sorted(plot_df["pair_sym"].unique())
    n_panels = len(sym_pairs)

    ncols = min(3, n_panels)
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
        squeeze=False,
        sharex=False,
        sharey=False,
    )
    axes = axes.ravel()

    directional_pairs = sorted(plot_df[pair_col].unique())
    pair_to_color = {
        p: plt.get_cmap("tab10")(i % 10)
        for i, p in enumerate(directional_pairs)
    }

    last_sc = None

    for ax, sym_pair in zip(axes, sym_pairs):
        sub = plot_df[plot_df["pair_sym"] == sym_pair].copy()

        # Error bars only make sense in aggregate mode
        if aggregate and show_yerr:
            ax.errorbar(
                sub["x"],
                sub["y"],
                yerr=sub["y_std"],
                fmt="none",
                capsize=3,
                color="gray",
                alpha=0.3,
            )

        if aggregate and show_xerr:
            ax.errorbar(
                sub["x"],
                sub["y"],
                xerr=sub["x_std"],
                fmt="none",
                capsize=3,
                color="gray",
                alpha=0.3,
            )

        # -----------------------
        # Coloring logic
        # -----------------------
        if color_by is None:
            ax.scatter(sub["x"], sub["y"], s=s, alpha=alpha)

        elif color_by == pair_col or color_by == "pair":
            for p in sorted(sub[pair_col].unique()):
                subp = sub[sub[pair_col] == p]
                ax.scatter(
                    subp["x"],
                    subp["y"],
                    s=s,
                    alpha=alpha,
                    color=pair_to_color[p],
                    label=p.replace("_to_", " → "),
                )

        elif color_by in sub.columns:
            # Decide: discrete legend or continuous colorbar
            is_numeric = pd.api.types.is_numeric_dtype(sub[color_by])
            nunique = sub[color_by].nunique()

            if (not is_numeric) or (nunique <= 10):
                vals = sorted(sub[color_by].dropna().unique())
                cmap_obj = plt.get_cmap("tab10")

                for i, val in enumerate(vals):
                    sub_val = sub[sub[color_by] == val]
                    ax.scatter(
                        sub_val["x"],
                        sub_val["y"],
                        color=cmap_obj(i % 10),
                        s=s,
                        alpha=alpha,
                        label=f"{color_by}={val}",
                    )

                if legend and ax is axes[0]:
                    ax.legend(title=color_by, fontsize=8)

            else:
                last_sc = ax.scatter(
                    sub["x"],
                    sub["y"],
                    c=sub[color_by],
                    cmap=cmap,
                    s=s,
                    alpha=alpha,
                )

        elif "color_val" in sub.columns:
            # aggregated numeric color column
            color_vals = sub["color_val"]
            nunique = color_vals.nunique()

            if nunique <= 10:
                vals = sorted(color_vals.dropna().unique())
                cmap_obj = plt.get_cmap("tab10")

                for i, val in enumerate(vals):
                    sub_val = sub[np.isclose(sub["color_val"], val)]
                    ax.scatter(
                        sub_val["x"],
                        sub_val["y"],
                        color=cmap_obj(i % 10),
                        s=s,
                        alpha=alpha,
                        label=f"{color_by}={val:g}",
                    )

                if legend and ax is axes[0]:
                    ax.legend(title=color_by, fontsize=8)

            else:
                last_sc = ax.scatter(
                    sub["x"],
                    sub["y"],
                    c=sub["color_val"],
                    cmap=cmap,
                    s=s,
                    alpha=alpha,
                )

        else:
            raise KeyError(f"Cannot color by '{color_by}'")

        if annotate:
            for _, r in sub.iterrows():
                label = f"G={int(r['G'])}, Y={int(r['Y'])}"
                if not aggregate and "seed" in r.index and pd.notna(r["seed"]):
                    label += f", seed={int(r['seed'])}"
                if color_by != "pair":
                    label = f"{r[pair_col].replace('_to_', '→')}\n{label}"
                ax.annotate(
                    label,
                    (r["x"], r["y"]),
                    xytext=(4, 4),
                    textcoords="offset points",
                    fontsize=8,
                )

        ax.set_title(sym_pair)
        ax.set_xlabel(xlabel or dist_col)
        ax.set_ylabel(ylabel or metric_col)

        if legend and (color_by == pair_col or color_by == "pair"):
            ax.legend(fontsize=8)

    for ax in axes[len(sym_pairs):]:
        ax.set_visible(False)

    if symmetric_axes:
        visible_axes = [ax for ax in axes if ax.get_visible()]
        xmin = min(ax.get_xlim()[0] for ax in visible_axes)
        xmax = max(ax.get_xlim()[1] for ax in visible_axes)
        ymin = min(ax.get_ylim()[0] for ax in visible_axes)
        ymax = max(ax.get_ylim()[1] for ax in visible_axes)

        lo = min(xmin, ymin)
        hi = max(xmax, ymax)

        for ax in visible_axes:
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)

    if title is not None:
        fig.suptitle(title, y=1.02)

    if color_by is not None and color_by not in [pair_col, "pair"
                                                 ] and last_sc is not None:
        cbar = fig.colorbar(last_sc, ax=axes[:len(sym_pairs)], shrink=0.9)
        cbar.set_label(color_by)

    plt.tight_layout()
    return fig, axes, plot_df

In [ ]:
fig, axes, df_summary = plot_distance_vs_metric_by_symmetric_pair(
    df,
    dist_col="D_wass_src_ft",
    metric_col="RMSE_annual",
    pair_col="pair",
    group_cols=("G", "Y"),
    xlabel=r"$D_{wass}^{S\to F}$",
    ylabel="RMSE (annual)",
    title="Transfer-learning RMSE vs source-to-finetune distance",
    color_by="G",
    annotate=True,
    min_g=5,
    min_y=3,
    figsize_per_panel=(6, 5),
    show_yerr=True,
    show_xerr=True,
)

plt.show()

In [ ]:
fig, axes, df_summary = plot_distance_vs_metric_by_symmetric_pair(
    df,
    dist_col="D_wass_src_ft",
    metric_col="D_wass_ft_holdout",
    pair_col="pair",
    group_cols=("G", "Y"),
    xlabel=r"$D_{wass}^{S\to F}$",
    ylabel=r"$D_{wass}^{F\to H}$",
    title=
    "Transfer-learning source-to-finetune vs finetune-to-holdout distance",
    color_by="G",
    annotate=False,
    min_g=5,
    min_y=3,
    figsize_per_panel=(6, 5),
    show_yerr=True,
    show_xerr=True,
    symmetric_axes=False,
)

plt.show()

### Two distances:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri


def plot_metric_over_two_distances(
    df,
    x_col="D_wass_ft_holdout",
    y_col="D_wass_src_ft",
    metric_col="RMSE_annual",
    group_cols=("pair", "G", "Y"),
    figsize=(7, 6),
    xlabel=None,
    ylabel=None,
    title=None,
    levels=12,
    cmap="viridis",
    vmin=None,
    vmax=None,
    annotate=False,
    annotate_cols=("G", "Y"),
    show_points=True,
    min_g=1,
    min_y=1,
    agg="mean",
):
    """
    Plot a triangulated contour surface of performance over two distances:
    x = D(F, H), y = D(S, F), z = performance metric.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    x_col : str
        Column for x-axis, e.g. D_wass_ft_holdout.
    y_col : str
        Column for y-axis, e.g. D_wass_src_ft.
    metric_col : str
        Metric to plot as contour value.
    group_cols : tuple[str]
        Columns used to aggregate repeated rows before plotting.
    figsize : tuple
        Figure size.
    xlabel, ylabel, title : str or None
        Axis labels and title.
    levels : int or array-like
        Contour levels passed to tricontourf.
    cmap : str
        Colormap.
    vmin, vmax : float or None
        Optional color scale limits.
    annotate : bool
        Whether to annotate points.
    annotate_cols : tuple[str]
        Columns used in annotations.
    show_points : bool
        Whether to overlay the aggregated points.
    min_g, min_y : int
        Minimum G and Y to include.
    agg : str
        Aggregation for metric and distances, usually "mean".

    Returns
    -------
    fig, ax, summary : matplotlib Figure, Axes, and aggregated dataframe
    """
    df_plot = df.copy()
    df_plot = df_plot[df_plot["G"] >= min_g]
    df_plot = df_plot[df_plot["Y"] >= min_y]

    needed = [x_col, y_col, metric_col, *group_cols]
    df_plot = df_plot.dropna(subset=needed).copy()
    if df_plot.empty:
        raise ValueError("No valid rows left after filtering.")

    summary = (df_plot.groupby(list(group_cols)).agg(
        x=(x_col, agg),
        y=(y_col, agg),
        z=(metric_col, agg),
        n=(metric_col, "size"),
    ).reset_index())

    if len(summary) < 3:
        raise ValueError("Need at least 3 points for triangulated contouring.")

    x = summary["x"].to_numpy(dtype=float)
    y = summary["y"].to_numpy(dtype=float)
    z = summary["z"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=figsize)

    triang = mtri.Triangulation(x, y)

    cf = ax.tricontourf(
        triang,
        z,
        levels=levels,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    cbar = fig.colorbar(cf, ax=ax)
    cbar.set_label(metric_col)

    # Optional contour lines
    cs = ax.tricontour(
        triang,
        z,
        levels=levels if not isinstance(levels, int) else 8,
        colors="k",
        linewidths=0.7,
        alpha=0.6,
    )
    ax.clabel(cs, fmt="%.2f", fontsize=8)

    if show_points:
        ax.scatter(x, y, s=35, alpha=0.8)

    if annotate:
        for _, r in summary.iterrows():
            label_parts = []
            for c in annotate_cols:
                if c in r.index:
                    label_parts.append(f"{c}={r[c]}")
            label = ", ".join(label_parts)
            ax.annotate(
                label,
                (r["x"], r["y"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )

    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    ax.set_title(title or f"{metric_col} over {x_col} and {y_col}")

    plt.tight_layout()
    return fig, ax, summary

In [ ]:
fig, ax, summary = plot_metric_over_two_distances(
    df=df,
    x_col="D_wass_src_ft",
    y_col="D_wass_ft_holdout",
    metric_col="RMSE_annual",
    group_cols=("pair", "G", "Y"),
    xlabel=r"$D_W(S, F)$",
    ylabel=r"$D_W(F, H)$",
    title="Annual RMSE over adaptation and representativity distances",
    cmap="viridis",
    levels=12,
    annotate=False,
    show_points=True,
)

In [ ]:
fig, ax, summary = plot_metric_over_two_distances(
    df=df,
    x_col="D_wass_src_ft",
    y_col="D_wass_ft_holdout",
    metric_col="GapClosed_annual",
    group_cols=("pair", "G", "Y"),
    xlabel=r"$D_W(S, F)$",
    ylabel=r"$D_W(F, H)$",
    title="Annual recovery over adaptation and representativity distances",
    cmap="viridis",
    levels=np.linspace(0, 1, 11),
    vmin=0,
    vmax=1,
    figsize=(14, 6),
)

In [ ]:
fig, ax, summary = plot_metric_over_two_distances(
    df=df,
    x_col="D_energy_src_ft",
    y_col="D_energy_ft_holdout",
    metric_col="GapClosed_annual",
    group_cols=("pair", "G", "Y"),
    xlabel=r"$D_E(S, F)$",
    ylabel=r"$D_E(F, H)$",
    title="Annual recovery over adaptation and representativity distances",
    cmap="viridis",
    levels=np.linspace(0, 1, 11),
    vmin=0,
    vmax=1,
    figsize=(14, 6),
)